# Classical AI Reasoning Agent

**Task:** given a knowledge base (facts + rules) and a query, decide whether the
query is *entailed* (True) or not (False).

**Pipeline:** natural language → semantic parser → logical representation →
classical inference engine → True/False. No generative model produces labels;
every prediction comes from a symbolic proof procedure.

| Input kind | Representation | Inference |
|---|---|---|
| Controlled English (RuleTaker style) | function-free Horn rules with negation | forward chaining to a fixpoint, negation-as-failure under the well-founded semantics (alternating fixpoint); backward chaining (SLDNF) as an alternative |
| First-order logic (FOLIO annotations) or free-form English | clauses (CNF, Skolemised) | resolution refutation with set-of-support, best-first clause selection, subsumption |
| Causal statements + counterfactual question (CounterBench) | Boolean structural causal model | abduction–action–prediction by exhaustive model checking |

The cells below contain the complete implementation, followed by experiments on
the public datasets, reasoning traces, and generation of `submission.csv`.

In [1]:
# ---- Configuration ---------------------------------------------------------
from pathlib import Path

# Path of an unseen test file (.csv/.json/.jsonl/.parquet). Column names are
# detected automatically (context/theory/premises, statement/question/query,
# id, optional label/flag and FOL annotations). If it does not exist, the
# submission is produced for the public evaluation sets instead.
TEST_PATH = Path("test.csv")
SUBMISSION_PATH = Path("submission.csv")
RULETAKER_SPLIT = "test"     # public split used for the RuleTaker experiment
RULETAKER_LIMIT = None       # None = whole split (151,911 examples)
PROVER_TIME_LIMIT = 0.3      # seconds per resolution proof

## Implementation: `logic` — First-order logic: terms, tolerant FOL parser, CNF + Skolemisation

In [2]:
"""First-order logic representation, a tolerant FOL parser, and CNF conversion.

Terms
    Var            -- a logical variable
    str            -- a constant symbol
    tuple          -- a function application ``(name, arg1, arg2, ...)``

Formulas are nested tuples:
    ("atom", pred, args)      ("not", f)          ("and", f1, f2, ...)
    ("or", f1, f2, ...)       ("imp", a, b)       ("iff", a, b)
    ("xor", a, b)             ("all", var, f)     ("ex", var, f)
    ("true",)                 ("false",)

Literals used by the provers are ``(positive, pred, args)`` triples and a
clause is a tuple of literals (a disjunction).
"""

import itertools
import re


class Var:
    """A logical variable. Identity is the integer id, the name is cosmetic."""

    __slots__ = ("id", "name")
    _counter = itertools.count()

    def __init__(self, name: str = "v"):
        self.id = next(Var._counter)
        self.name = name

    def __repr__(self) -> str:
        return f"{self.name.upper()}{self.id}"

    def __eq__(self, other):
        return self is other

    def __hash__(self):
        return self.id

    def __lt__(self, other):
        return str(self) < str(other)


def is_var(term) -> bool:
    return type(term) is Var


def term_str(term) -> str:
    if type(term) is tuple:
        return f"{term[0]}({', '.join(term_str(a) for a in term[1:])})"
    return str(term)


def atom_str(pred: str, args: tuple) -> str:
    if pred == "=" and len(args) == 2:
        return f"{term_str(args[0])} = {term_str(args[1])}"
    if not args:
        return pred
    return f"{pred}({', '.join(term_str(a) for a in args)})"


def literal_str(lit) -> str:
    positive, pred, args = lit
    return ("" if positive else "¬") + atom_str(pred, args)


def clause_str(clause) -> str:
    """Print a clause with its variables renamed X, Y, Z, ... for readability."""
    if not clause:
        return "□"
    names: dict[Var, str] = {}

    def rename(t):
        if type(t) is Var:
            if t not in names:
                i = len(names)
                names[t] = "XYZWUV"[i] if i < 6 else f"X{i}"
            return names[t]
        if type(t) is tuple:
            return (t[0], *(rename(a) for a in t[1:]))
        return t

    return " ∨ ".join(
        literal_str((pos, pred, tuple(rename(a) for a in args))) for pos, pred, args in clause
    )


def formula_str(f) -> str:
    op = f[0]
    if op == "atom":
        return atom_str(f[1], f[2])
    if op == "true":
        return "⊤"
    if op == "false":
        return "⊥"
    if op == "not":
        return "¬" + formula_str(f[1])
    if op in ("all", "ex"):
        q = "∀" if op == "all" else "∃"
        return f"{q}{f[1]!r} {formula_str(f[2])}"
    symbol = {"and": " ∧ ", "or": " ∨ ", "imp": " → ", "iff": " ↔ ", "xor": " ⊕ "}[op]
    return "(" + symbol.join(formula_str(g) for g in f[1:]) + ")"


# ---------------------------------------------------------------------------
# Formula constructors
# ---------------------------------------------------------------------------


def Atom(pred: str, *args):
    return ("atom", pred, tuple(args))


def Not(f):
    return ("not", f)


def And(*fs):
    fs = [f for f in fs if f != ("true",)]
    if not fs:
        return ("true",)
    return fs[0] if len(fs) == 1 else ("and", *fs)


def Or(*fs):
    fs = [f for f in fs if f != ("false",)]
    if not fs:
        return ("false",)
    return fs[0] if len(fs) == 1 else ("or", *fs)


def Implies(a, b):
    return ("imp", a, b)


def ForAll(var, f):
    return ("all", var, f)


def Exists(var, f):
    return ("ex", var, f)


# ---------------------------------------------------------------------------
# Tolerant parser for FOLIO-style first-order logic strings
# ---------------------------------------------------------------------------

_TOKEN_RE = re.compile(
    r"\s*(?:"
    r"(?P<op><->|<=>|[-—–]>|=>|⟷|↔|⇔|→|⇒|⊕|∧|∨|¬|∀|∃|≠|!=|=|&|\||~|\^|\(|\)|,|\[|\]|\{|\})"
    r"|(?P<id>\w[\w'.\-]*)"
    r"|(?P<junk>\S))"
)

_OP_ALIASES = {
    "<->": "↔", "<=>": "↔", "⟷": "↔", "⇔": "↔", "->": "→", "—>": "→", "–>": "→",
    "=>": "→", "⇒": "→", "&": "∧", "^": "∧",
    "|": "∨", "~": "¬", "!=": "≠", "[": "(", "]": ")", "{": "(", "}": ")",
}
_WORD_OPS = {
    "forall": "∀", "exists": "∃", "and": "∧", "or": "∨", "not": "¬",
    "implies": "→", "iff": "↔", "xor": "⊕",
}
_FREE_VARIABLE_NAMES = re.compile(r"^[a-z]\d*$")


class FOLParseError(ValueError):
    pass


def _tokenize(text: str) -> list[str]:
    tokens = []
    for match in _TOKEN_RE.finditer(text):
        if match.group("op"):
            tok = match.group("op")
            tokens.append(_OP_ALIASES.get(tok, tok))
        elif match.group("id"):
            tok = match.group("id").rstrip(".-")
            if tok.lower() in _WORD_OPS and tok.lower() == tok:
                tokens.append(_WORD_OPS[tok])
            elif tok == "v" and tokens and tokens[-1] == ")":
                tokens.append("∨")  # "P(a) v Q(a)": a term cannot follow ")"
            elif tok:
                tokens.append(tok)
        # Unknown junk characters (stray quotes, em-dashes) are ignored.
    return tokens


def _balance(tokens: list[str]) -> list[str]:
    """Repair unbalanced parentheses, a frequent annotation error in FOLIO."""
    depth = 0
    out = []
    for tok in tokens:
        if tok == "(":
            depth += 1
        elif tok == ")":
            if depth == 0:
                continue
            depth -= 1
        out.append(tok)
    return out + [")"] * depth


class _FOLParser:
    # Precedence (loosest first): ↔, →, ⊕, ∨, ∧, ¬/quantifiers.
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0
        self.scope: list[dict[str, Var]] = []

    def peek(self, offset=0):
        i = self.pos + offset
        return self.tokens[i] if i < len(self.tokens) else None

    def take(self, expected=None):
        tok = self.peek()
        if tok is None or (expected is not None and tok != expected):
            raise FOLParseError(f"expected {expected!r}, got {tok!r}")
        self.pos += 1
        return tok

    def parse(self):
        f = self.iff()
        if self.peek() is not None:
            raise FOLParseError(f"trailing tokens: {self.tokens[self.pos:]}")
        return f

    def iff(self):
        left = self.imp()
        while self.peek() == "↔":
            self.take()
            left = ("iff", left, self.imp())
        return left

    def imp(self):
        left = self.xor()
        if self.peek() == "→":
            self.take()
            return ("imp", left, self.imp())
        return left

    def xor(self):
        left = self.disj()
        while self.peek() == "⊕":
            self.take()
            left = ("xor", left, self.disj())
        return left

    def disj(self):
        parts = [self.conj()]
        while self.peek() == "∨":
            self.take()
            parts.append(self.conj())
        return parts[0] if len(parts) == 1 else ("or", *parts)

    def conj(self):
        parts = [self.unary()]
        while self.peek() == "∧":
            self.take()
            parts.append(self.unary())
        return parts[0] if len(parts) == 1 else ("and", *parts)

    def unary(self):
        tok = self.peek()
        if tok == "¬":
            self.take()
            return ("not", self.unary())
        if tok in ("∀", "∃"):
            self.take()
            names = []
            while self.peek() not in (None, "(", "¬", "∀", "∃") and self._is_name(
                self.peek()
            ):
                # "∀x ∀y" or "∀x y" or "∀x,y"
                names.append(self.take())
                if self.peek() == ",":
                    self.take()
                if self.peek(1) == "(" and self.peek() not in ("∀", "∃"):
                    break
            op = "all" if tok == "∀" else "ex"
            if not names:
                # "∃(P(x) ∧ Q(x))": bind the free variables of the scope.
                if self.peek() != "(":
                    raise FOLParseError("quantifier without variable")
                free: dict[str, Var] = {}
                body = _bind_free(self.iff(), free)
                for v in reversed(free.values()):
                    body = (op, v, body)
                return body
            variables = [Var(n) for n in names]
            self.scope.append(dict(zip(names, variables)))
            # Quantifier scope extends as far right as possible.
            body = self.iff()
            self.scope.pop()
            for v in reversed(variables):
                body = (op, v, body)
            return body
        if tok == "(":
            self.take()
            f = self.iff()
            if self.peek() == ")":
                self.take()
            return f
        return self.atomic()

    @staticmethod
    def _is_name(tok):
        return tok is not None and re.match(r"^\w", tok) is not None

    def lookup(self, name):
        for frame in reversed(self.scope):
            if name in frame:
                return frame[name]
        return None

    def term(self):
        name = self.take()
        if not self._is_name(name):
            raise FOLParseError(f"bad term {name!r}")
        if self.peek() == "(":
            self.take()
            args = [self.term()]
            while self.peek() == ",":
                self.take()
                args.append(self.term())
            if self.peek() == ")":
                self.take()
            return (name, *args)
        bound = self.lookup(name)
        if bound is not None:
            return bound
        if _FREE_VARIABLE_NAMES.match(name):
            return ("$free", name)
        return name

    def atomic(self):
        tok = self.peek()
        if not self._is_name(tok):
            raise FOLParseError(f"unexpected token {tok!r}")
        if self.peek(1) == "(":
            name = self.take()
            self.take("(")
            args = []
            if self.peek() != ")":
                args.append(self.term())
                while self.peek() == ",":
                    self.take()
                    args.append(self.term())
            if self.peek() == ")":
                self.take()
            atom = ("atom", name, tuple(args))
        else:
            left = self.term()
            if self.peek() in ("=", "≠"):
                op = self.take()
                right = self.term()
                atom = ("atom", "=", (left, right))
                return atom if op == "=" else ("not", atom)
            if type(left) is tuple and left[0] == "$free":
                left = left[1]
            atom = ("atom", str(left), ())
        if self.peek() in ("=", "≠"):
            # A predicate application used as a term: P(x) = c (rare).
            op = self.take()
            right = self.term()
            eq = ("atom", "=", ((atom[1], *atom[2]), right))
            return eq if op == "=" else ("not", eq)
        return atom


def _bind_free(f, free: dict[str, Var]):
    """Replace placeholder free variables by shared universally bound Vars."""
    op = f[0]
    if op == "atom":
        return ("atom", f[1], tuple(_bind_term(t, free) for t in f[2]))
    if op in ("all", "ex"):
        return (op, f[1], _bind_free(f[2], free))
    if op in ("true", "false"):
        return f
    return (op, *(_bind_free(g, free) for g in f[1:]))


def _bind_term(t, free):
    if type(t) is tuple:
        if t[0] == "$free":
            if t[1] not in free:
                free[t[1]] = Var(t[1])
            return free[t[1]]
        return (t[0], *(_bind_term(a, free) for a in t[1:]))
    return t


def parse_fol(text: str, free_quantifier: str = "all"):
    """Parse one FOL formula.

    Free variables are closed with ``free_quantifier``: universally for
    premises, existentially for goals ("Cost(gre, x) ∧ Cheaper(x, 300)").
    """
    tokens = _balance(_tokenize(text))
    if not tokens:
        raise FOLParseError("empty formula")
    f = _FOLParser(tokens).parse()
    free: dict[str, Var] = {}
    f = _bind_free(f, free)
    for v in free.values():
        f = (free_quantifier, v, f)
    return f


# ---------------------------------------------------------------------------
# Clause normal form
# ---------------------------------------------------------------------------


def _nnf(f, positive=True):
    """Negation normal form with →, ↔, ⊕ eliminated."""
    op = f[0]
    if op == "atom":
        return f if positive else ("not", f)
    if op == "true":
        return f if positive else ("false",)
    if op == "false":
        return f if positive else ("true",)
    if op == "not":
        return _nnf(f[1], not positive)
    if op == "and":
        parts = [_nnf(g, positive) for g in f[1:]]
        return ("and", *parts) if positive else ("or", *parts)
    if op == "or":
        parts = [_nnf(g, positive) for g in f[1:]]
        return ("or", *parts) if positive else ("and", *parts)
    if op == "imp":
        return _nnf(("or", ("not", f[1]), f[2]), positive)
    if op == "iff":
        a, b = f[1], f[2]
        if positive:
            return _nnf(("and", ("imp", a, b), ("imp", b, a)))
        return _nnf(("xor", a, b))
    if op == "xor":
        a, b = f[1], f[2]
        if positive:
            return ("and", ("or", _nnf(a), _nnf(b)), ("or", _nnf(a, False), _nnf(b, False)))
        return _nnf(("iff", a, b))
    if op in ("all", "ex"):
        flip = {"all": "ex", "ex": "all"}
        return ((op if positive else flip[op]), f[1], _nnf(f[2], positive))
    raise ValueError(f"unknown operator {op}")


class _Skolemizer:
    def __init__(self, prefix="sk"):
        self.counter = itertools.count()
        self.prefix = prefix

    def run(self, f, env: dict, universals: tuple):
        op = f[0]
        if op == "atom":
            return ("atom", f[1], tuple(_subst_term(t, env) for t in f[2]))
        if op == "not":
            return ("not", self.run(f[1], env, universals))
        if op in ("and", "or"):
            return (op, *(self.run(g, env, universals) for g in f[1:]))
        if op == "all":
            fresh = Var(f[1].name)
            return self.run(f[2], {**env, f[1]: fresh}, universals + (fresh,))
        if op == "ex":
            name = f"{self.prefix}{next(self.counter)}"
            sk = (name, *universals) if universals else name
            return self.run(f[2], {**env, f[1]: sk}, universals)
        return f


def _subst_term(t, env):
    if type(t) is Var:
        return env.get(t, t)
    if type(t) is tuple:
        return (t[0], *(_subst_term(a, env) for a in t[1:]))
    return t


def _to_literal(f):
    if f[0] == "not":
        return (False, f[1][1], f[1][2])
    return (True, f[1], f[2])


_definition_counter = itertools.count()


def _cnf_clauses(f, limit: int, extra: list) -> list[frozenset]:
    """Clauses for an NNF, quantifier-free formula.

    Distribution is used while the result stays small; larger disjuncts are
    replaced by fresh definition atoms (Tseitin), whose defining clauses are
    appended to ``extra``. This preserves satisfiability.
    """
    op = f[0]
    if op in ("atom", "not"):
        return [frozenset([_to_literal(f)])]
    if op == "true":
        return []
    if op == "false":
        return [frozenset()]
    if op == "and":
        out = []
        for g in f[1:]:
            out.extend(_cnf_clauses(g, limit, extra))
        return out
    if op == "or":
        subs = [_cnf_clauses(g, limit, extra) for g in f[1:]]
        size = 1
        for sub in subs:
            size *= max(1, len(sub))
        if size > limit:
            renamed = []
            for g, sub in zip(f[1:], subs):
                if len(sub) > 1:
                    args = tuple(sorted(free_vars_of_formula(g), key=lambda v: v.id))
                    name = f"def{next(_definition_counter)}"
                    for c in sub:
                        extra.append(c | {(False, name, args)})
                    sub = [frozenset([(True, name, args)])]
                renamed.append(sub)
            subs = renamed
        result = [frozenset()]
        for sub in subs:
            result = [a | b for a in result for b in sub]
        return result
    raise ValueError(op)


def _is_tautology(clause) -> bool:
    return any((not p, q, a) in clause for (p, q, a) in clause)


def to_clauses(f, skolemizer: _Skolemizer | None = None, limit: int = 64):
    """Convert a closed formula into a list of clauses (tuples of literals)."""
    skolemizer = skolemizer or _Skolemizer()
    g = skolemizer.run(_nnf(f), {}, ())
    extra: list = []
    clauses = _cnf_clauses(g, limit, extra) + extra
    out = []
    seen = set()
    for c in clauses:
        if _is_tautology(c):
            continue
        key = tuple(sorted(c, key=repr))
        if key not in seen:
            seen.add(key)
            out.append(key)
    return out


def free_vars_of_formula(f, bound=frozenset()):
    op = f[0]
    if op == "atom":
        out = set()
        for t in f[2]:
            out |= _term_vars(t)
        return out - bound
    if op in ("all", "ex"):
        return free_vars_of_formula(f[2], bound | {f[1]})
    if op in ("true", "false"):
        return set()
    out = set()
    for g in f[1:]:
        out |= free_vars_of_formula(g, bound)
    return out


def _term_vars(t):
    if type(t) is Var:
        return {t}
    if type(t) is tuple:
        out = set()
        for a in t[1:]:
            out |= _term_vars(a)
        return out
    return set()


def close_universally(f):
    for v in free_vars_of_formula(f):
        f = ("all", v, f)
    return f


def _map_formula(f, pred_map, const_map):
    op = f[0]
    if op == "atom":
        return ("atom", pred_map.get((f[1], len(f[2])), f[1]),
                tuple(_map_term(t, const_map) for t in f[2]))
    if op in ("all", "ex"):
        return (op, f[1], _map_formula(f[2], pred_map, const_map))
    if op in ("true", "false"):
        return f
    return (op, *(_map_formula(g, pred_map, const_map) for g in f[1:]))


def _map_term(t, const_map):
    if type(t) is tuple:
        return (t[0], *(_map_term(a, const_map) for a in t[1:]))
    if type(t) is str:
        return const_map.get(t, t)
    return t


def _symbol_counts(f, preds, consts):
    op = f[0]
    if op == "atom":
        preds[(f[1], len(f[2]))] = preds.get((f[1], len(f[2])), 0) + 1
        for t in f[2]:
            _const_counts(t, consts)
    elif op in ("all", "ex"):
        _symbol_counts(f[2], preds, consts)
    elif op not in ("true", "false"):
        for g in f[1:]:
            _symbol_counts(g, preds, consts)


def _const_counts(t, consts):
    if type(t) is tuple:
        for a in t[1:]:
            _const_counts(a, consts)
    elif type(t) is str:
        consts[t] = consts.get(t, 0) + 1


def _norm_symbol(name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", name.lower())


def _similar(a: str, b: str) -> bool:
    import difflib

    x, y = _norm_symbol(a), _norm_symbol(b)
    if not x or not y:
        return False
    if x == y:
        return True
    shorter, longer = sorted((x, y), key=len)
    if len(shorter) >= 4 and longer.startswith(shorter) and len(longer) - len(shorter) <= 3:
        return True
    if len(shorter) >= 4 and longer.endswith(shorter) and len(longer) - len(shorter) <= 1:
        return True
    return len(shorter) >= 5 and difflib.SequenceMatcher(None, x, y).ratio() >= 0.86


def repair_symbols(premises, goal):
    """Merge near-duplicate symbols (typos / inconsistent naming).

    A symbol that occurs only once across the problem is renamed to a
    sufficiently similar symbol of the same kind and arity that occurs more
    often; a symbol used only by the goal is renamed to a similar premise
    symbol. Exact case-insensitive matches are always merged.
    """
    preds, consts = {}, {}
    for f in premises:
        _symbol_counts(f, preds, consts)
    premise_syms = set(preds) | set(consts)
    _symbol_counts(goal, preds, consts)

    def build(counts, key_name, key_arity):
        mapping = {}
        items = sorted(counts.items(), key=lambda kv: -kv[1])
        for sym, n in items:
            name, arity = key_name(sym), key_arity(sym)
            for other, m in items:
                if other == sym or key_arity(other) != arity or other in mapping:
                    continue
                oname = key_name(other)
                exact = _norm_symbol(oname) == _norm_symbol(name)
                if exact and (m > n or (m == n and oname < name)):
                    mapping[sym] = oname
                    break
                goal_only = sym not in premise_syms and other in premise_syms
                if (goal_only or (n == 1 and m >= 2)) and _similar(name, oname):
                    mapping[sym] = oname
                    break
        return mapping

    pred_map = build(preds, lambda s: s[0], lambda s: s[1])
    const_map = build(consts, lambda s: s, lambda s: 0)
    const_map = {k: v for k, v in const_map.items() if not k.isdigit() and not v.isdigit()}
    if not pred_map and not const_map:
        return list(premises), goal, {}
    fixed = [_map_formula(f, pred_map, const_map) for f in premises]
    changes = {**{f"{k[0]}/{k[1]}": v for k, v in pred_map.items()}, **const_map}
    return fixed, _map_formula(goal, pred_map, const_map), changes


def predicates_of(f, out=None):
    out = set() if out is None else out
    op = f[0]
    if op == "atom":
        out.add((f[1], len(f[2])))
    elif op in ("all", "ex"):
        predicates_of(f[2], out)
    elif op not in ("true", "false"):
        for g in f[1:]:
            predicates_of(g, out)
    return out

## Implementation: `prover` — Resolution theorem prover (unification, set-of-support, subsumption)

In [3]:
"""Refutation theorem prover for first-order clauses.

Algorithm: binary resolution + factoring inside a given-clause loop.

* Set-of-support: only clauses descending from the negated goal are selected
  as given clauses; the knowledge-base clauses act as the "usable" side.
  This keeps the search goal-directed and avoids "proving" queries from an
  inconsistent annotation.
* Clause selection is best-first (uniform-cost style) on clause weight
  (number of symbols), interleaved with oldest-first picks for fairness.
* Forward subsumption and tautology deletion prune the search space.
* Equality, when present, is axiomatised (reflexivity, symmetry,
  transitivity, and predicate substitutivity).
"""

import heapq
import time
from dataclasses import dataclass, field


# ---------------------------------------------------------------------------
# Unification
# ---------------------------------------------------------------------------


def walk(t, s):
    while type(t) is Var and t in s:
        t = s[t]
    return t


def occurs(v, t, s) -> bool:
    t = walk(t, s)
    if t is v:
        return True
    if type(t) is tuple:
        return any(occurs(v, a, s) for a in t[1:])
    return False


def unify(a, b, s):
    """Most general unifier extending substitution ``s`` (or None)."""
    a = walk(a, s)
    b = walk(b, s)
    if a is b:
        return s
    if type(a) is Var:
        if occurs(a, b, s):
            return None
        s = dict(s)
        s[a] = b
        return s
    if type(b) is Var:
        if occurs(b, a, s):
            return None
        s = dict(s)
        s[b] = a
        return s
    if type(a) is tuple and type(b) is tuple:
        if len(a) != len(b) or a[0] != b[0]:
            return None
        for x, y in zip(a[1:], b[1:]):
            s = unify(x, y, s)
            if s is None:
                return None
        return s
    return s if a == b else None


def unify_args(xs, ys, s):
    if len(xs) != len(ys):
        return None
    for x, y in zip(xs, ys):
        s = unify(x, y, s)
        if s is None:
            return None
    return s


def substitute(t, s):
    t = walk(t, s)
    if type(t) is tuple:
        return (t[0], *(substitute(a, s) for a in t[1:]))
    return t


def match(pattern, target, s):
    """One-way matching: bind only variables of ``pattern``."""
    if type(pattern) is Var:
        if pattern in s:
            return s if s[pattern] == target else None
        s = dict(s)
        s[pattern] = target
        return s
    if type(pattern) is tuple:
        if type(target) is not tuple or len(pattern) != len(target):
            return None
        if pattern[0] != target[0]:
            return None
        for x, y in zip(pattern[1:], target[1:]):
            s = match(x, y, s)
            if s is None:
                return None
        return s
    return s if pattern == target else None


# ---------------------------------------------------------------------------
# Clause utilities
# ---------------------------------------------------------------------------


def clause_vars(clause):
    out = []
    seen = set()

    def visit(t):
        if type(t) is Var:
            if t not in seen:
                seen.add(t)
                out.append(t)
        elif type(t) is tuple:
            for a in t[1:]:
                visit(a)

    for _, _, args in clause:
        for a in args:
            visit(a)
    return out


def rename(clause):
    vs = clause_vars(clause)
    if not vs:
        return clause
    s = {v: Var(v.name) for v in vs}
    return tuple((p, q, tuple(substitute(a, s) for a in args)) for p, q, args in clause)


def apply_clause(lits, s):
    out = []
    for p, q, args in lits:
        lit = (p, q, tuple(substitute(a, s) for a in args))
        if lit not in out:
            out.append(lit)
    return tuple(out)


def is_tautology(clause) -> bool:
    for p, q, a in clause:
        if p and (False, q, a) in clause:
            return True
        if p and q == "=" and len(a) == 2 and a[0] == a[1]:
            return True
    return False


def _term_weight(t):
    if type(t) is tuple:
        return 1 + sum(_term_weight(a) for a in t[1:])
    return 1


def weight(clause) -> int:
    return sum(1 + sum(_term_weight(a) for a in args) for _, _, args in clause)


def canonical(clause):
    """Variant-invariant key (variables numbered by first occurrence)."""
    mapping = {}

    def canon(t):
        if type(t) is Var:
            if t not in mapping:
                mapping[t] = f"?{len(mapping)}"
            return mapping[t]
        if type(t) is tuple:
            return (t[0], *(canon(a) for a in t[1:]))
        return t

    lits = sorted(clause, key=lambda lit: (lit[1], lit[0], repr(lit[2])))
    return tuple((p, q, tuple(canon(a) for a in args)) for p, q, args in lits)


def subsumes(c, d) -> bool:
    """True if clause c θ-subsumes clause d."""
    if len(c) > len(d):
        return False

    def search(i, s):
        if i == len(c):
            return True
        p, q, args = c[i]
        for p2, q2, args2 in d:
            if p2 == p and q2 == q and len(args2) == len(args):
                s2 = s
                for x, y in zip(args, args2):
                    s2 = match(x, y, s2)
                    if s2 is None:
                        break
                if s2 is not None and search(i + 1, s2):
                    return True
        return False

    return search(0, {})


# ---------------------------------------------------------------------------
# Prover
# ---------------------------------------------------------------------------


@dataclass
class ProofResult:
    proved: bool
    reason: str
    steps: int = 0
    clauses: int = 0
    proof: list[str] = field(default_factory=list)


def equality_axioms(clauses):
    preds = {}
    for c in clauses:
        for _, q, args in c:
            if q != "=":
                preds[q] = len(args)
    x, y, z = Var("x"), Var("y"), Var("z")
    axioms = [
        ((True, "=", (x, x)),),
        ((False, "=", (x, y)), (True, "=", (y, x))),
        ((False, "=", (x, y)), (False, "=", (y, z)), (True, "=", (x, z))),
    ]
    for q, n in preds.items():
        for i in range(n):
            a = [Var("a") for _ in range(n)]
            b = list(a)
            u, v = Var("u"), Var("v")
            a[i], b[i] = u, v
            axioms.append(((False, "=", (u, v)), (False, q, tuple(a)), (True, q, tuple(b))))
    return axioms


def _depth(t):
    if type(t) is tuple:
        return 1 + max((_depth(a) for a in t[1:]), default=0)
    return 0


def remove_pure(axioms, sos):
    """Drop clauses containing a literal whose complement occurs nowhere.

    Such clauses can never take part in a refutation (pure literal rule).
    """
    while True:
        signs = {(p, q) for c in axioms + sos for p, q, _ in c}
        keep = lambda c: all((not p, q) in signs for p, q, _ in c)  # noqa: E731
        new_axioms = [c for c in axioms if keep(c)]
        new_sos = [c for c in sos if keep(c)]
        if len(new_axioms) == len(axioms) and len(new_sos) == len(sos):
            return axioms, sos
        axioms, sos = new_axioms, new_sos


class Prover:
    def __init__(self, max_steps=3000, max_clauses=30000, time_limit=2.0, age_ratio=5,
                 max_literals=8, max_depth=3):
        self.max_steps = max_steps
        self.max_clauses = max_clauses
        self.time_limit = time_limit
        self.age_ratio = age_ratio
        self.max_literals = max_literals
        self.max_depth = max_depth

    def refute(self, axioms, sos) -> ProofResult:
        """Search for the empty clause from ``axioms`` ∪ ``sos``."""
        deadline = time.perf_counter() + self.time_limit
        axioms = [c for c in axioms if not is_tautology(c)]
        sos = [c for c in sos if not is_tautology(c)]
        if any(q == "=" for c in axioms + sos for _, q, _ in c):
            axioms = axioms + equality_axioms(axioms + sos)
        axioms, sos = remove_pure(axioms, sos)
        if not sos:
            return ProofResult(False, "search exhausted")

        store: list[tuple] = []  # clause id -> clause
        parents: list = []
        seen = set()
        usable_by_lit: dict[tuple, list[int]] = {}
        usable_ids: list[int] = []
        weight_heap: list = []
        age_heap: list = []
        selected = set()

        def add(clause, origin):
            key = canonical(clause)
            if key in seen:
                return None
            seen.add(key)
            cid = len(store)
            store.append(clause)
            parents.append(origin)
            return cid

        subsume_index: dict[tuple, list[int]] = {}

        def make_usable(cid):
            usable_ids.append(cid)
            clause = store[cid]
            for i, (p, q, _) in enumerate(clause):
                usable_by_lit.setdefault((p, q), []).append((cid, i))
            if clause:
                # A subsumer must share all its literal keys with the subsumed
                # clause, so indexing it under one key finds it exactly once.
                p, q, _ = clause[0]
                subsume_index.setdefault((p, q), []).append(cid)

        for c in axioms:
            cid = add(c, ("axiom",))
            if cid is not None:
                make_usable(cid)

        def push(cid):
            w = weight(store[cid])
            heapq.heappush(weight_heap, (w, cid))
            heapq.heappush(age_heap, (cid, cid))

        for c in sos:
            if not c:
                return ProofResult(True, "empty clause in input", proof=["□"])
            cid = add(c, ("goal",))
            if cid is not None:
                push(cid)

        def forward_subsumed(clause):
            preds = {(p, q) for p, q, _ in clause}
            n = len(clause)
            for key in preds:
                for cid in subsume_index.get(key, ()):
                    other = store[cid]
                    if len(other) <= n and all((a, b) in preds for a, b, _ in other):
                        if subsumes(other, clause):
                            return True
            return False

        steps = 0
        while weight_heap or age_heap:
            if steps >= self.max_steps or len(store) >= self.max_clauses:
                return ProofResult(False, "resource limit", steps, len(store))
            if time.perf_counter() > deadline:
                return ProofResult(False, "time limit", steps, len(store))
            heap = age_heap if (steps % self.age_ratio == self.age_ratio - 1) else weight_heap
            if not heap:
                heap = weight_heap or age_heap
            _, gid = heapq.heappop(heap)
            if gid in selected:
                continue
            selected.add(gid)
            given = store[gid]
            steps += 1
            if forward_subsumed(given):
                continue
            make_usable(gid)
            g = rename(given)

            new = []
            # Binary resolution against every usable clause (including itself).
            for i, (p, q, args) in enumerate(g):
                for oid, j in list(usable_by_lit.get((not p, q), ())):
                    other = rename(store[oid]) if oid != gid else rename(g)
                    _, _, oargs = other[j]
                    s = unify_args(args, oargs, {})
                    if s is None:
                        continue
                    lits = g[:i] + g[i + 1 :] + other[:j] + other[j + 1 :]
                    new.append((apply_clause(lits, s), ("res", gid, oid)))
            # Factoring of the given clause.
            for i in range(len(g)):
                for j in range(i + 1, len(g)):
                    if g[i][0] == g[j][0] and g[i][1] == g[j][1]:
                        s = unify_args(g[i][2], g[j][2], {})
                        if s is not None:
                            new.append((apply_clause(g[:j] + g[j + 1 :], s), ("fac", gid)))

            for clause, origin in new:
                if not clause:
                    cid = len(store)
                    store.append(clause)
                    parents.append(origin)
                    proof = self._trace(store, parents, cid)
                    return ProofResult(True, "refutation found", steps, len(store), proof)
                if len(clause) > self.max_literals or any(
                    _depth(a) > self.max_depth for _, _, args in clause for a in args
                ):
                    continue
                if is_tautology(clause) or forward_subsumed(clause):
                    continue
                cid = add(clause, origin)
                if cid is not None:
                    push(cid)
        return ProofResult(False, "search exhausted", steps, len(store))

    @staticmethod
    def _trace(store, parents, cid) -> list[str]:
        order = []
        visited = set()

        def visit(i):
            if i in visited:
                return
            visited.add(i)
            origin = parents[i]
            for p in origin[1:]:
                visit(p)
            order.append(i)

        visit(cid)
        lines = []
        for i in order:
            origin = parents[i]
            how = origin[0]
            if len(origin) > 1:
                how += f"({', '.join(map(str, origin[1:]))})"
            lines.append(f"[{i}] {clause_str(store[i])}    <- {how}")
        return lines


def prove_formula(premises, goal, prover: Prover | None = None) -> ProofResult:
    """Does the conjunction of FOL ``premises`` entail ``goal``?"""

    prover = prover or Prover()
    sk = _Skolemizer()
    axioms = []
    for f in premises:
        axioms.extend(to_clauses(f, sk))
    negated = to_clauses(("not", goal), _Skolemizer("skg"))
    return prover.refute(axioms, negated)

## Implementation: `datalog` — Forward chaining (well-founded NAF) and backward chaining (SLDNF)

In [4]:
"""Rule engine for function-free Horn rules with negation.

A rule is ``body -> head`` where body literals may be negated and the head is a
single literal (positive, or negative = "strong"/explicit negation).

Two semantics are provided:

* ``cwa`` (closed world): a negated body literal is *negation as failure*.
  Models are computed with the well-founded semantics by the alternating
  fixpoint (Van Gelder), which equals the perfect model for stratified
  programs. Each inner step is forward chaining to a least fixpoint.
* ``owa`` (open world): negated literals are ordinary facts about a
  complementary predicate, so ``¬p`` is only true if it is stated/derived.

Literals are ``(positive, pred, args)``; variables are ``Var`` objects.
"""

from dataclasses import dataclass, field



@dataclass(frozen=True)
class Rule:
    body: tuple  # tuple of literals
    head: tuple  # a single literal
    text: str = ""

    def __str__(self):
        names: dict[Var, str] = {}

        def show(lit):
            positive, pred, args = lit
            shown = []
            for a in args:
                if type(a) is Var:
                    if a not in names:
                        i = len(names)
                        names[a] = "XYZWUV"[i] if i < 6 else f"X{i}"
                    a = names[a]
                shown.append(a)
            return literal_str((positive, pred, tuple(shown)))

        body = [show(b) for b in self.body]
        head = show(self.head)
        return " ∧ ".join(body) + " → " + head if body else head


def _ground(args, binding):
    return tuple(binding.get(a, a) if type(a) is Var else a for a in args)


def _match_args(pattern, fact_args, binding):
    for p, f in zip(pattern, fact_args):
        if type(p) is Var:
            bound = binding.get(p)
            if bound is None:
                binding = {**binding, p: f}
            elif bound != f:
                return None
        elif p != f:
            return None
    return binding


@dataclass
class Model:
    true: set  # set of (pred, args) atoms that are true
    possible: set  # superset: atoms not known to be false
    neg: set = field(default_factory=set)  # explicitly derived ¬atoms
    why: dict = field(default_factory=dict)  # literal -> (rule, body instances)
    iterations: int = 0
    rule_firings: int = 0

    def holds(self, positive, pred, args, cwa=True):
        atom = (pred, args)
        if positive:
            return atom in self.true
        if (pred, args) in self.neg:
            return True
        return cwa and atom not in self.possible


class ForwardChainer:
    def __init__(self, rules, domain=(), cwa=True, max_facts=200_000):
        self.rules = list(rules)
        self.cwa = cwa
        self.max_facts = max_facts
        consts = set(domain)
        for r in self.rules:
            for _, _, args in (*r.body, r.head):
                consts.update(a for a in args if type(a) is not Var)
        self.domain = sorted(consts)
        self.firings = 0

    # -- core least-fixpoint computation --------------------------------------

    def _least_model(self, naf_reference, record=None):
        """Least fixpoint, NAF literals evaluated against ``naf_reference``.

        ``naf_reference`` is a set of atoms; ``not a`` holds iff a ∉ it.
        In OWA mode negative literals are separate "¬" predicates instead.
        """
        facts: dict[tuple, set] = {}  # (positive, pred) -> set(args)
        new_by_key: dict[tuple, set] = {}

        def add(lit, rule, body):
            key = (lit[0], lit[1])
            bucket = facts.setdefault(key, set())
            if lit[2] in bucket:
                return False
            bucket.add(lit[2])
            new_by_key.setdefault(key, set()).add(lit[2])
            if record is not None and (lit not in record):
                record[lit] = (rule, body)
            return True

        pending = []
        for r in self.rules:
            if not r.body:
                head = r.head
                if any(type(a) is Var for a in head[2]):
                    for c in self.domain:
                        g = tuple(c if type(a) is Var else a for a in head[2])
                        add((head[0], head[1], g), r, ())
                else:
                    add(head, r, ())
            else:
                pending.append(r)

        split = []
        for r in pending:
            pos = [b for b in r.body if b[0] or not self.cwa]
            neg = [b for b in r.body if not b[0] and self.cwa]
            pos.sort(key=lambda b: -len(b[2]))
            split.append((r, pos, neg))

        first = True
        while True:
            changed = False
            delta = new_by_key
            new_by_key = {}
            for r, pos, neg in split:
                # Semi-naive: some positive literal must match a fact that was
                # new in the previous round; rules without positive literals
                # only need to be tried in the first round.
                for binding in self._join(pos, facts, delta, first):
                    for full in self._bindings_for_neg(neg, binding):
                        ok = True
                        for b in neg:
                            atom = (b[1], _ground(b[2], full))
                            if atom in naf_reference:
                                ok = False
                                break
                        if not ok:
                            continue
                        head_args = _ground(r.head[2], full)
                        if any(type(a) is Var for a in head_args):
                            heads = [
                                tuple(c if type(a) is Var else a for a in head_args)
                                for c in self.domain
                            ]
                        else:
                            heads = [head_args]
                        for h in heads:
                            self.firings += 1
                            body_inst = tuple(
                                (b[0], b[1], _ground(b[2], full)) for b in r.body
                            )
                            if add((r.head[0], r.head[1], h), r, body_inst):
                                changed = True
                if sum(len(v) for v in facts.values()) > self.max_facts:
                    raise RuntimeError("fact limit exceeded")
            first = False
            if not changed:
                break
        return facts

    def _bindings_for_neg(self, neg, binding):
        unbound = []
        for b in neg:
            for a in b[2]:
                if type(a) is Var and a not in binding and a not in unbound:
                    unbound.append(a)
        if not unbound:
            yield binding
            return

        def rec(i, bnd):
            if i == len(unbound):
                yield bnd
                return
            for c in self.domain:
                yield from rec(i + 1, {**bnd, unbound[i]: c})

        yield from rec(0, binding)

    def _join(self, pos, facts, delta, first):
        """Yield bindings satisfying all positive literals.

        Semi-naive evaluation: at least one literal must match a fact from
        ``delta`` (facts new in the previous round). On the first round,
        ``delta`` contains every fact.
        """
        if not pos:
            if first:
                yield {}
            return
        for pivot in range(len(pos)):
            key = (pos[pivot][0], pos[pivot][1])
            if not delta.get(key):
                continue
            yield from self._join_rec(pos, 0, {}, facts, delta, pivot)

    def _join_rec(self, pos, i, binding, facts, delta, pivot):
        if i == len(pos):
            yield binding
            return
        lit = pos[i]
        key = (lit[0], lit[1])
        if i == pivot:
            source = delta.get(key, ())
        elif i < pivot:
            # Literals before the pivot use old facts only (avoid duplicates).
            source = facts.get(key, set()) - delta.get(key, set())
        else:
            source = facts.get(key, ())
        pattern = lit[2]
        if all(type(a) is not Var or a in binding for a in pattern):
            g = _ground(pattern, binding)
            if g in source:
                yield from self._join_rec(pos, i + 1, binding, facts, delta, pivot)
            return
        for fact_args in list(source):
            if len(fact_args) != len(pattern):
                continue
            b2 = _match_args(pattern, fact_args, binding)
            if b2 is not None:
                yield from self._join_rec(pos, i + 1, b2, facts, delta, pivot)

    # -- semantics --------------------------------------------------------------

    def model(self, max_rounds=50) -> Model:
        if not self.cwa:
            record = {}
            facts = self._least_model(set(), record)
            true = {(p, a) for (s, p), args in facts.items() if s for a in args}
            neg = {(p, a) for (s, p), args in facts.items() if not s for a in args}
            return Model(true, true | neg, neg, record, 1, self.firings)

        # Alternating fixpoint for the well-founded semantics.
        has_naf = any(not b[0] for r in self.rules for b in r.body)
        everything = _Universe()
        under_ref = everything  # NAF false everywhere -> under-approximation
        rounds = 0
        prev_true = None
        true_atoms = set()
        possible = set()
        record = {}
        while rounds < max_rounds:
            rounds += 1
            rec = {}
            facts_t = self._least_model(under_ref if rounds > 1 else everything, rec)
            true_atoms = {(p, a) for (s, p), args in facts_t.items() if s for a in args}
            neg_atoms = {(p, a) for (s, p), args in facts_t.items() if not s for a in args}
            record = rec
            if not has_naf:
                possible = true_atoms
                break
            facts_u = self._least_model(true_atoms)
            possible = {(p, a) for (s, p), args in facts_u.items() if s for a in args}
            if true_atoms == prev_true:
                break
            prev_true = true_atoms
            under_ref = possible
        return Model(true_atoms, possible, neg_atoms, record, rounds, self.firings)


class _Universe:
    """A set-like object containing every atom (NAF fails everywhere)."""

    def __contains__(self, item):
        return True


def explain(model: Model, lit, depth=0, seen=None, max_depth=25) -> list[str]:
    """Human-readable proof tree for a derived literal."""
    seen = set() if seen is None else seen
    pad = "  " * depth
    positive, pred, args = lit
    key = (positive, pred, args)
    entry = model.why.get(key)
    text = literal_str(lit)
    if entry is None:
        if not positive:
            return [f"{pad}{text}   [negation as failure: {atom_str(pred, args)} is not derivable]"]
        return [f"{pad}{text}   [not derivable]"]
    rule, body = entry
    if not body:
        return [f"{pad}{text}   [fact]"]
    if key in seen or depth > max_depth:
        return [f"{pad}{text}   [see above]"]
    seen.add(key)
    lines = [f"{pad}{text}   [rule: {rule}]"]
    for b in body:
        lines.extend(explain(model, b, depth + 1, seen, max_depth))
    return lines


# ---------------------------------------------------------------------------
# Backward chaining (SLDNF with loop checking), an alternative strategy.
# ---------------------------------------------------------------------------


class BackwardChainer:
    """Goal-directed proof search over the same rules.

    Negated goals use negation as failure (CWA) or explicit negation (OWA).
    Ground goals are tabled to avoid repeated work and infinite loops.
    """

    def __init__(self, rules, domain=(), cwa=True, max_depth=60):
        self.cwa = cwa
        self.max_depth = max_depth
        self.by_head: dict[tuple, list[Rule]] = {}
        consts = set(domain)
        for r in rules:
            self.by_head.setdefault((r.head[0], r.head[1]), []).append(r)
            for _, _, args in (*r.body, r.head):
                consts.update(a for a in args if type(a) is not Var)
        self.domain = sorted(consts)
        self.table: dict[tuple, bool] = {}
        self.calls = 0

    def prove(self, lit) -> bool:
        return self._prove(lit, frozenset(), 0)

    def _prove(self, lit, stack, depth) -> bool:
        positive, pred, args = lit
        if not positive and self.cwa:
            # Negation as failure. In a stratified program the negated goal
            # does not depend on its ancestors, so it starts a fresh loop check.
            return not self._prove_atom((True, pred, args), frozenset(), depth)
        return self._prove_atom(lit, stack, depth)

    def _prove_atom(self, lit, stack, depth) -> bool:
        self.calls += 1
        if lit in self.table:
            return self.table[lit]
        if lit in stack or depth > self.max_depth:
            return False
        stack = stack | {lit}
        result = False
        for r in self.by_head.get((lit[0], lit[1]), ()):
            binding = _match_args(r.head[2], lit[2], {})
            if binding is None:
                continue
            if self._prove_body(list(r.body), binding, stack, depth + 1):
                result = True
                break
        if result or not stack - {lit}:
            self.table[lit] = result
        return result

    def _prove_body(self, body, binding, stack, depth) -> bool:
        if not body:
            return True
        # Prefer a literal whose arguments are already ground.
        body.sort(key=lambda b: (not b[0], sum(type(a) is Var and a not in binding for a in b[2])))
        lit, rest = body[0], body[1:]
        args = _ground(lit[2], binding)
        unbound = [a for a in args if type(a) is Var]
        if not unbound:
            return self._prove((lit[0], lit[1], args), stack, depth) and self._prove_body(
                rest, binding, stack, depth
            )
        for values in _product(self.domain, len(set(unbound))):
            b2 = dict(binding)
            for v, c in zip(dict.fromkeys(unbound), values):
                b2[v] = c
            g = _ground(lit[2], b2)
            if self._prove((lit[0], lit[1], g), stack, depth) and self._prove_body(
                list(rest), b2, stack, depth
            ):
                return True
        return False


def _product(domain, n):
    if n == 0:
        yield ()
        return
    for c in domain:
        for rest in _product(domain, n - 1):
            yield (c, *rest)

## Implementation: `english` — Natural language -> logic: compositional semantic parser

In [5]:
"""Rule-based semantic parser: restricted English -> logic.

The parser is compositional rather than template based. A sentence is
segmented into a *condition* and a *conclusion* (``if/when/whenever ...``,
``... if ...``, ``all/every/no/some N ...``, generic plurals), each side is
split into coordinated clauses, and every clause is analysed as

    SUBJECT  VERB-PHRASE

where the subject is an entity (``the bald eagle``, ``Anne``), an indefinite
or pronoun (``someone``, ``they``, ``it``) bound to a rule variable, and the
verb phrase is a copula + adjective/noun complement, a (negated) transitive or
intransitive verb, or a ``has``/``can`` construction.

Output items are ``Rule`` objects (Horn rules / facts, possibly with negated
literals) or general FOL formulas for disjunctive/existential content.
"""

import re
from dataclasses import dataclass, field


COPULA = {"is", "are", "am", "was", "were", "be", "being", "been"}
AUX = {"does", "do", "did"}
MODALS = {"can", "could", "will", "would", "shall", "should", "may", "might", "must"}
NEGATORS = {"not", "never", "no"}
DETERMINERS = {"a", "an", "the"}
QUANTIFIERS = {"all", "every", "each", "any", "no", "some"}
RELATIVE = {"who", "that", "which", "whom"}
PRONOUNS = {"they", "it", "he", "she", "them", "him", "her", "itself", "themselves",
            "himself", "herself"}
INDEFINITES = {"someone", "something", "somebody", "anyone", "anything", "anybody",
               "everyone", "everything", "everybody", "one"}
GENERIC_NOUNS = {"thing", "things", "people", "person", "persons", "someone", "something",
                 "somebody", "one", "ones", "individual", "individuals", "everyone",
                 "everything", "everybody", "anyone", "anything", "anybody", "guy",
                 "guys", "entity", "entities", "creature", "creatures", "being",
                 "beings", "object", "objects", "animal", "animals", "man", "men",
                 "woman", "women"}
FILLERS = {"always", "usually", "normally", "generally", "typically", "often", "also",
           "really", "very", "quite", "rather", "pretty", "definitely", "certainly",
           "surely", "indeed", "just", "too", "actually", "truly", "sometimes",
           "absolutely", "extremely", "somewhat", "fairly", "likewise",
           "still", "necessarily", "therefore", "thus", "hence", "so"}
COPULA_PHRASES = [
    ("tends to be", "is"), ("tend to be", "are"), ("seems to be", "is"),
    ("seem to be", "are"), ("is going to be", "is"), ("are going to be", "are"),
    ("will be", "is"), ("must be", "is"), ("should be", "is"), ("would be", "is"),
    ("is sure to be", "is"), ("are sure to be", "are"), ("happens to be", "is"),
    ("happen to be", "are"), ("is considered", "is"), ("are considered", "are"),
    ("is likely to be", "is"), ("are likely to be", "are"), ("can be", "is"),
    ("turns out to be", "is"), ("turn out to be", "are"), ("ends up being", "is"),
    ("is also", "is"), ("are also", "are"), ("is definitely", "is"),
]
_COPULA_PHRASE_WORDS: dict[str, list] = {}
for _phrase, _repl in sorted(COPULA_PHRASES, key=lambda x: -len(x[0].split())):
    _COPULA_PHRASE_WORDS.setdefault(_phrase.split()[0], []).append((_phrase.split(), _repl))
CONTRACTIONS = {
    "doesn't": "does not", "don't": "do not", "didn't": "did not", "isn't": "is not",
    "aren't": "are not", "wasn't": "was not", "weren't": "were not",
    "can't": "can not", "cannot": "can not", "won't": "will not",
    "shouldn't": "should not", "wouldn't": "would not", "couldn't": "could not",
    "hasn't": "has not", "haven't": "have not", "it's": "it is", "they're": "they are",
    "he's": "he is", "she's": "she is", "someone's": "someone is",
}
PREPOSITIONS = {"of", "to", "with", "at", "in", "on", "for", "from", "by", "about",
                "into", "than", "as", "like", "near", "under", "over"}


_FUNCTION_WORDS = (COPULA | AUX | MODALS | DETERMINERS | QUANTIFIERS | PRONOUNS | INDEFINITES
                   | {"if", "when", "whenever", "there", "everyone", "nobody", "nothing",
                      "either", "neither", "only"})


class ParseFailure(ValueError):
    pass


# ---------------------------------------------------------------------------
# Lexical helpers
# ---------------------------------------------------------------------------


def lemma_verb(word: str) -> str:
    """Third-person singular / plural verb forms -> one canonical form."""
    w = word.lower()
    if w in ("is", "are", "am", "was", "were", "be"):
        return "be"
    if w in ("has", "have", "had"):
        return "have"
    if w in ("does", "do", "did"):
        return "do"
    if w.endswith("ies") and len(w) > 4:
        return w[:-3] + "y"
    if w.endswith(("ches", "shes", "sses", "xes", "zzes", "oes")):
        return w[:-2]
    if w.endswith("s") and not w.endswith(("ss", "us", "is")) and len(w) > 2:
        return w[:-1]
    return w


def singular(noun: str) -> str:
    n = noun.lower()
    irregular = {"people": "person", "men": "man", "women": "woman", "mice": "mouse",
                 "geese": "goose", "children": "child", "teeth": "tooth", "feet": "foot",
                 "oxen": "ox", "cacti": "cactus", "wolves": "wolf", "knives": "knife",
                 "lives": "life", "leaves": "leaf", "halves": "half", "calves": "calf"}
    if n in irregular:
        return irregular[n]
    if n.endswith("ies") and len(n) > 4:
        return n[:-3] + "y"
    if n.endswith(("ches", "shes", "sses", "xes", "zes")):
        return n[:-2]
    if n.endswith("s") and not n.endswith(("ss", "us", "is", "ous")) and len(n) > 3:
        return n[:-1]
    return n


def looks_plural(word: str) -> bool:
    w = word.lower()
    return w in ("people", "men", "women", "mice", "children", "geese") or (
        w.endswith("s") and not w.endswith(("ss", "us", "ous", "is")) and len(w) > 3
    )


def entity_name(words: list[str]) -> str:
    return "_".join(w.lower() for w in words)


# ---------------------------------------------------------------------------
# Tokens
# ---------------------------------------------------------------------------


@dataclass
class Tok:
    text: str  # lower-case
    cap: bool  # capitalised in the source


def normalize(sentence: str) -> list[Tok]:
    s = sentence.strip()
    s = s.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    s = re.sub(r"[.!?;:]+\s*$", "", s)
    words = re.findall(r"[A-Za-z0-9]+(?:['\-][A-Za-z0-9]+)*|,", s)
    expanded: list[Tok] = []
    for w in words:
        low = w.lower()
        if low in CONTRACTIONS:
            for part in CONTRACTIONS[low].split():
                expanded.append(Tok(part, False))
        elif low.endswith("n't"):
            expanded.append(Tok(low[:-3], w[0].isupper()))
            expanded.append(Tok("not", False))
        else:
            expanded.append(Tok(low, w[0].isupper()))
    if expanded:
        expanded[0].cap = expanded[0].cap and expanded[0].text not in _FUNCTION_WORDS
    # Multi-word copula paraphrases ("tends to be" -> "is").
    out: list[Tok] = []
    i = 0
    while i < len(expanded):
        for phrase, repl in _COPULA_PHRASE_WORDS.get(expanded[i].text, ()):
            if [t.text for t in expanded[i : i + len(phrase)]] == phrase:
                out.append(Tok(repl, False))
                i += len(phrase)
                break
        else:
            if expanded[i].text not in FILLERS:
                out.append(expanded[i])
            i += 1
    return out


# ---------------------------------------------------------------------------
# Parsing context
# ---------------------------------------------------------------------------


@dataclass
class Ctx:
    """Variable bindings for one sentence."""

    main: object = None  # the rule's primary variable (or entity)
    others: list = field(default_factory=list)
    last_entity: str | None = None
    vars_: dict = field(default_factory=dict)

    def variable(self, key="x"):
        if key not in self.vars_:
            self.vars_[key] = Var(key)
        return self.vars_[key]


@dataclass
class Clause:
    subject: object
    lits: list  # list of literals (positive, pred, args)
    kind: str = "conj"  # "conj", "or" (disjunction of lits), "nor"
    copular: bool = False
    negated: bool = False
    verb: str | None = None
    indefinite: bool = False


# ---------------------------------------------------------------------------
# The parser
# ---------------------------------------------------------------------------


class EnglishParser:
    def __init__(self):
        self.known_verbs: set[str] = set()
        self.known_entities: set[str] = set()
        self.last_entity: str | None = None
        self._pending_verbs: set[str] = set()
        self._lemma_cache: tuple = (set(), set())
        self._witnesses = 0

    # -- public API -------------------------------------------------------------

    def fork(self) -> "EnglishParser":
        """A copy sharing the learnt lexicon but not mutating this parser."""
        other = EnglishParser()
        other.known_verbs = set(self.known_verbs)
        other.known_entities = set(self.known_entities)
        other.last_entity = self.last_entity
        other._witnesses = self._witnesses
        return other

    @staticmethod
    def is_controlled(items) -> bool:
        """True if every item uses only the core controlled-language fragment.

        Fallback constructions (prepositional / multi-word predicates,
        equality, disjunctions) signal free-form text, for which the closed
        world assumption is not appropriate.
        """
        for it in items:
            if not isinstance(it, Rule):
                return False
            for _, pred, _ in (*it.body, it.head):
                if "_" in pred or pred == "=":
                    return False
        return True

    def parse_text(self, text: str):
        """Parse a multi-sentence context. Returns (items, failures)."""
        items, failures = [], []
        for sentence in split_sentences(text):
            try:
                items.extend(self.parse_sentence(sentence))
            except ParseFailure as exc:
                failures.append((sentence, str(exc)))
        return items, failures

    def parse_sentence(self, sentence: str):
        toks = normalize(sentence)
        if not toks:
            return []
        self._pending_verbs = set()
        items = self._sentence(toks)
        # Verbs are only learnt from clauses of successfully parsed sentences
        # that produced binary relations.
        binary = {q for it in items if isinstance(it, Rule)
                  for _, q, a in (*it.body, it.head) if len(a) == 2}
        self.known_verbs |= {v for v in self._pending_verbs if lemma_verb(v) in binary}
        for item in items:
            item_text = sentence.strip()
            if isinstance(item, Rule):
                object.__setattr__(item, "text", item_text)
        return items

    # -- sentence level ---------------------------------------------------------

    def _sentence(self, toks):
        words = [t.text for t in toks]
        ctx = Ctx(last_entity=self.last_entity)
        if words[0] in ("if", "when", "whenever", "provided") or words[:2] == ["as", "long"]:
            start = 1 if words[0] != "as" else 3
            body_toks, head_toks = self._split_condition(toks[start:])
            return self._conditional(body_toks, head_toks, ctx)
        if "if" in words[1:]:
            k = words.index("if", 1)
            if words[k - 1] not in ("even", "as", "only"):
                head = toks[:k]
                if head and head[-1].text == ",":
                    head = head[:-1]
                body = toks[k + 1 :]
                if self._starts_with_entity(head) or head[0].text in PRONOUNS | INDEFINITES:
                    return self._conditional(body, head, ctx)
                # Generic head: "Big people are red if they are young."
                rules = self._quantified(head, ctx, implicit=head[0].text not in QUANTIFIERS)
                extra = self._condition(body, ctx)
                return [Rule(r.body + tuple(alt), r.head) for r in rules
                        if isinstance(r, Rule) for alt in extra]
        if len(toks) > 3 and toks[0].cap and words[1] == "," and words[2] in RELATIVE:
            # Appositive: "Alan, who is round and red, is blue" -> facts.
            toks = [toks[0]] + toks[3:]
            words = [t.text for t in toks]
        if words[0] == "there" and len(words) > 2 and words[1] in ("is", "are", "exists", "exist"):
            return self._existential(toks[2:], ctx)
        if words[0] in QUANTIFIERS or words[0] in ("everyone", "everything", "everybody",
                                                   "nobody", "nothing", "anyone", "anything"):
            return self._quantified(toks, ctx)
        if words[0] in ("a", "an"):
            return self._quantified([Tok("every", False)] + toks[1:], ctx)
        if words[0] in INDEFINITES:
            return self._quantified(toks, ctx)
        if not self._starts_with_entity(toks):
            return self._quantified(toks, ctx, implicit=True)
        clauses = self._conjunction(toks, ctx, None)
        return self._facts(clauses, ctx)

    def _split_condition(self, toks):
        words = [t.text for t in toks]
        if "then" in words:
            k = words.index("then")
            body = toks[:k]
            if body and body[-1].text == ",":
                body = body[:-1]
            return body, toks[k + 1 :]
        # "If A, B": split at the last comma whose right side starts a clause.
        commas = [i for i, w in enumerate(words) if w == ","]
        for k in reversed(commas):
            right = toks[k + 1 :]
            if right and (right[0].text in PRONOUNS | INDEFINITES | {"the"} or right[0].cap):
                return toks[:k], right
        if commas:
            k = commas[-1]
            return toks[:k], toks[k + 1 :]
        # "If someone is red they are big": split before a pronoun subject.
        for k in range(1, len(words)):
            if words[k] in ("they", "it", "he", "she") and k + 1 < len(words) and (
                words[k + 1] in COPULA | AUX | MODALS or self._is_verb_word(words[k + 1])
            ):
                return toks[:k], toks[k:]
        raise ParseFailure("cannot split condition / conclusion")

    def _conditional(self, body_toks, head_toks, ctx):
        alternatives = self._condition(body_toks, ctx)
        heads = self._conjunction(head_toks, ctx, None, head=True)
        rules = []
        for alt in alternatives:
            rules.extend(self._make_rules(alt, heads, ctx))
        return rules

    def _condition(self, toks, ctx):
        """Condition -> list of alternatives, each a list of literals (DNF)."""
        words = [t.text for t in toks]
        groups = _split_top(toks, "or") if "or" in words and "either" not in words else [toks]
        alternatives = []
        prev = None
        for group in groups:
            clauses = self._conjunction(group, ctx, prev)
            prev = clauses[-1] if clauses else prev
            lits = []
            for c in clauses:
                if c.kind == "or":
                    raise ParseFailure("disjunction inside condition clause")
                lits.extend(c.lits)
            alternatives.append(lits)
        return alternatives

    def _make_rules(self, body_lits, head_clauses, ctx):
        rules = []
        for c in head_clauses:
            if c.kind == "or":
                return [("formula", _implication(body_lits, c))]
            for lit in c.lits:
                rules.append(Rule(tuple(body_lits), lit))
        return rules

    def _facts(self, clauses, ctx):
        items = []
        for c in clauses:
            if isinstance(c.subject, str):
                self.last_entity = c.subject
            if c.kind == "or":
                items.append(("formula", ("or", *(_lit_formula(lit) for lit in c.lits))))
                continue
            for lit in c.lits:
                items.append(Rule((), lit))
        return items

    def _existential(self, toks, ctx):
        """'There is a big dog that is red' -> skolem constant facts."""
        restrictor, vp = self._split_np_vp(toks, ctx, allow_no_vp=True)
        lits = restrictor
        if vp:
            for c in vp:
                lits = lits + c.lits
        self._witnesses += 1
        name = f"witness_{self._witnesses}"
        x = ctx.main
        return [Rule((), (p, q, tuple(name if a is x else a for a in args))) for p, q, args in lits]

    def _quantified(self, toks, ctx, implicit=False):
        words = [t.text for t in toks]
        quant = None
        if not implicit:
            first = words[0]
            if first in ("everyone", "everything", "everybody", "anyone", "anything"):
                quant = "all"
                toks = [Tok("all", False), Tok("things", False)] + toks[1:]
            elif first in ("nobody", "nothing"):
                quant = "no"
                toks = [Tok("no", False), Tok("things", False)] + toks[1:]
            elif first in ("someone", "something", "somebody"):
                quant = "all"
                toks = [Tok("all", False), Tok("things", False)] + toks[1:]
            else:
                quant = {"every": "all", "each": "all", "any": "all"}.get(first, first)
            toks = toks[1:]
        else:
            quant = "all"
        if quant == "some":
            return self._existential(toks, ctx)
        restrictor, heads = self._split_np_vp(toks, ctx)
        if quant == "no":
            negated = []
            for c in heads:
                if c.kind != "conj" or len(c.lits) != 1:
                    return [("formula", _negated_universal(restrictor, c))]
                p, q, a = c.lits[0]
                negated.append(Clause(c.subject, [(not p, q, a)]))
            heads = negated
        return self._make_rules(restrictor, heads, ctx)

    def _split_np_vp(self, toks, ctx, allow_no_vp=False):
        """Split 'ADJ* NOUN [who VP] VP' into restrictor literals + head clauses."""
        x = ctx.variable("x")
        ctx.main = x
        # Prefer splitting at a copula/auxiliary, then at a lexical verb.
        candidates = [i for i, t in enumerate(toks) if i > 0 and t.text in COPULA | AUX | MODALS]
        candidates += [
            i for i, t in enumerate(toks)
            if i > 0 and i not in candidates
            and (self._is_verb_word(t.text) or t.text in ("has", "have"))
        ]
        errors = []
        for k in candidates:
            try:
                restrictor = self._restrictor(toks[:k], x, ctx)
                heads = self._conjunction(toks[k:], ctx, Clause(x, [], copular=False),
                                          head=True, subject=x)
                return restrictor, heads
            except ParseFailure as exc:
                errors.append(str(exc))
        if allow_no_vp:
            return self._restrictor(toks, x, ctx), []
        raise ParseFailure("cannot split noun phrase and verb phrase: " + "; ".join(errors[:2]))

    def _restrictor(self, toks, x, ctx):
        """'kind , smart things who are big' -> literals over x."""
        words = [t.text for t in toks]
        rel = next((i for i, w in enumerate(words) if w in RELATIVE), None)
        lits = []
        rel_lits = []
        if rel is not None:
            rc = toks[rel + 1 :]
            clauses = self._conjunction(rc, ctx, Clause(x, []), subject=x)
            for c in clauses:
                if c.kind != "conj":
                    raise ParseFailure("disjunction in relative clause")
                rel_lits.extend(c.lits)
            toks = toks[:rel]
            words = words[:rel]
        # Post-nominal modifiers: "people with red hair" -> skip; "dogs in the park"
        if any(w in PREPOSITIONS for w in words[1:]):
            k = next(i for i, w in enumerate(words) if i > 0 and w in PREPOSITIONS)
            pp = toks[k:]
            toks, words = toks[:k], words[:k]
            pred = "_".join(t.text for t in pp if t.text not in DETERMINERS and t.text != ",")
            if not toks:
                raise ParseFailure("empty noun phrase")
            lits.append((True, f"{singular(words[-1])}_{pred}", (x,)))
        words = [w for w in words if w not in (",", "and", "a", "an", "the")]
        if not words:
            raise ParseFailure("empty restrictor")
        for w in words[:-1]:
            if w in COPULA | AUX | MODALS or w in RELATIVE:
                raise ParseFailure(f"unexpected {w!r} in noun phrase")
            if w == "not":
                raise ParseFailure("negation in noun phrase")
            lits.append((True, w, (x,)))
        head = words[-1]
        if head in COPULA | AUX | MODALS:
            raise ParseFailure("verb in noun phrase")
        if head not in GENERIC_NOUNS:
            lits.append((True, singular(head), (x,)))
        return lits + rel_lits

    # -- clause level -------------------------------------------------------------

    def _starts_with_entity(self, toks):
        if not toks:
            return False
        w = toks[0].text
        if w == "the" or w in PRONOUNS:
            return True
        if w in self.known_entities:
            return True
        if toks[0].cap:
            # Sentence-initial capitals are ambiguous ("Cold people are ...").
            k = 1
            while k < len(toks) and toks[k].cap:
                k += 1
            if k + 1 < len(toks) and toks[k].text in ("and", ",") and (
                toks[k + 1].cap or toks[k + 1].text == "the"
            ):
                return True  # coordinated subject: "Bob and Anne ..."
            return k < len(toks) and self._starts_vp(toks[k:])
        # Lower-case proper names: accept if directly followed by a verb.
        if len(toks) > 1 and (toks[1].text in COPULA | AUX | MODALS) and not looks_plural(w):
            return True
        return False

    def _conjunction(self, toks, ctx, prev, head=False, subject=None):
        """Coordinated clauses with subject/verb ellipsis."""
        parts = _split_coordination(toks)
        clauses = []
        pending_subjects = []  # coordinated subjects: "the cat and the dog are big"
        for part in parts:
            if not part:
                continue
            if not clauses and subject is None:
                np = self._subject(part, ctx)
                if np is not None and not np[1]:
                    pending_subjects.append(np[0])
                    continue
                if pending_subjects and part[0].cap and (np is None or not np[1]):
                    # Plural verb after coordinated names: "Bob and Anne like ..."
                    k = 1
                    while k < len(part) and part[k].cap:
                        k += 1
                    np = (entity_name([t.text for t in part[:k]]), part[k:])
                if pending_subjects and np is not None and np[1]:
                    for s in [*pending_subjects, np[0]]:
                        clauses.append(self._verb_phrase(np[1], s, ctx))
                    prev = clauses[-1]
                    pending_subjects = []
                    continue
            c = self._clause(part, ctx, prev, subject)
            clauses.append(c)
            prev = c
            subject = None
        if pending_subjects:
            raise ParseFailure("coordinated noun phrase without verb phrase")
        if not clauses:
            raise ParseFailure("empty clause")
        return clauses

    def _clause(self, toks, ctx, prev, forced_subject=None):
        words = [t.text for t in toks]
        if forced_subject is not None:
            subj, rest = forced_subject, toks
        else:
            parsed = self._subject_with_vp(toks, ctx)
            if parsed is None:
                if prev is None:
                    raise ParseFailure(f"no subject in {' '.join(words)!r}")
                subj = prev.subject
                rest = toks
                if prev.copular and words[0] in ("not", "never") and (
                    len(words) == 1 or words[1] not in COPULA | AUX | MODALS
                ):
                    rest = [Tok("is", False)] + rest
                elif prev.indefinite and not self._starts_vp(rest):
                    # "is a cold, round man": adjectives continue the noun phrase.
                    rest = [Tok("is", False), Tok("a", False)] + rest
                elif not self._starts_vp(rest):
                    # Ellipsis: "is red and (is) big", "likes the cat and (likes) the dog"
                    if prev.copular:
                        rest = [Tok("is", False)] + ([Tok("not", False)] if prev.negated
                                                     and words[0] not in NEGATORS else []) + rest
                    elif prev.verb:
                        rest = [Tok(prev.verb, False)] + rest
                    else:
                        rest = [Tok("is", False)] + rest
            else:
                subj, rest = parsed
        return self._verb_phrase(rest, subj, ctx)

    def _known_lemmas(self):
        if len(self._lemma_cache[0]) != len(self.known_verbs):
            self._lemma_cache = (set(self.known_verbs), {lemma_verb(v) for v in self.known_verbs})
        return self._lemma_cache[1]

    def _subject_with_vp(self, toks, ctx):
        parsed = self._subject(toks, ctx)
        if parsed is not None and not parsed[1]:
            return None  # a bare noun phrase is an elided object, not a subject
        return parsed

    def _starts_vp(self, toks):
        if not toks:
            return False
        w = toks[0].text
        return w in COPULA | AUX | MODALS | {"has", "have", "not", "never"} or (
            self._is_verb_word(w) and (len(toks) > 1 or w in self.known_verbs)
        )

    def _is_verb_word(self, w):
        if w in self.known_verbs or w in self._known_lemmas():
            return True
        return (w.endswith("s") and not w.endswith(("ss", "us", "is", "ous")) and len(w) > 2
                and w not in GENERIC_NOUNS and w not in ("this", "has", "was", "its", "his",
                                                         "hers", "theirs", "yes", "less"))

    def _subject(self, toks, ctx):
        """Return (term, remaining tokens) or None if the clause has no subject."""
        w = toks[0].text
        if w in PRONOUNS:
            if ctx.main is not None:
                return ctx.main, toks[1:]
            if ctx.last_entity is not None:
                return ctx.last_entity, toks[1:]
            x = ctx.variable("x")
            ctx.main = x
            return x, toks[1:]
        if w in INDEFINITES:
            if ctx.main is None:
                ctx.main = ctx.variable("x")
            return ctx.main, toks[1:]
        if w in ("a", "an") and len(toks) > 2:
            k = self._np_end(toks, 1)
            if k is not None:
                if ctx.main is None:
                    ctx.main = ctx.variable("x")
                return ctx.main, toks[k:]
        if w == "the":
            k = self._np_end(toks, 1)
            if k is None:
                return None
            name = entity_name([t.text for t in toks[1:k] if t.text != ","])
            self.known_entities.add(name)
            ctx.last_entity = name
            return name, toks[k:]
        if toks[0].cap or w in self.known_entities:
            k = 1
            while k < len(toks) and toks[k].cap and toks[k].text not in COPULA:
                k += 1
            if k == len(toks) or self._starts_vp(toks[k:]):
                name = entity_name([t.text for t in toks[:k]])
                self.known_entities.add(name)
                ctx.last_entity = name
                return name, toks[k:]
            # fall through: maybe a multi-word lower-case name
        # Lower-case name directly followed by a verb: "bob is big".
        if len(toks) > 1 and toks[1].text in COPULA | AUX | MODALS and w not in NEGATORS:
            if w in ("and", "or", ",", "then") or w in PREPOSITIONS:
                return None
            if ctx.main is not None and w in ("someone", "something"):
                return ctx.main, toks[1:]
            if w not in self.known_entities and not toks[0].cap and len(toks) > 2:
                # Could be an adjective in an elided clause ("red is"?); accept as name.
                pass
            self.known_entities.add(w)
            return w, toks[1:]
        return None

    def _np_end(self, toks, start):
        """End index of a noun phrase starting at ``start`` (exclusive)."""
        for k in range(start + 1, len(toks) + 1):
            if k == len(toks):
                return k
            if self._starts_vp(toks[k:]) and toks[k].text not in ("not", "never"):
                return k
            if toks[k].text in RELATIVE:
                return None
        return None

    def _object(self, toks, ctx):
        """Parse an object noun phrase. Returns (term or None, unary preds)."""
        words = [t.text for t in toks if t.text != ","]
        if not words:
            return None
        if len(words) == 1 and (words[0] in PRONOUNS):
            if words[0] in ("itself", "themselves", "himself", "herself"):
                return ctx.main, []
            if ctx.main is not None:
                return ctx.main, []
            return ctx.last_entity, []
        if len(words) == 1 and words[0] in INDEFINITES:
            if ctx.main is None:
                ctx.main = ctx.variable("x")
                return ctx.main, []
            y = ctx.variable(f"y{len(ctx.vars_)}")
            return y, []
        if words[0] == "the":
            name = entity_name(words[1:])
            self.known_entities.add(name)
            return name, []
        if toks[0].cap or words[0] in self.known_entities or len(words) == 1:
            name = entity_name(words)
            self.known_entities.add(name)
            return name, []
        return None

    def _verb_phrase(self, toks, subj, ctx) -> Clause:
        words = [t.text for t in toks]
        if not words:
            raise ParseFailure("missing verb phrase")
        neg = False
        i = 0
        w = words[0]
        if w in MODALS:
            i += 1
            if i < len(words) and words[i] in NEGATORS:
                neg = True
                i += 1
            if i < len(words) and words[i] == "be":
                words = ["is"] + words[i + 1 :]
                toks = [Tok("is", False)] + toks[i + 1 :]
                i = 0
                w = "is"
            else:
                rest = toks[i:]
                c = self._verb_phrase(rest, subj, ctx)
                if neg:
                    c.lits = [(not p, q, a) for p, q, a in c.lits]
                    c.negated = True
                return c
        if w in COPULA:
            i = 1
            if i < len(words) and words[i] in NEGATORS and words[i] != "no":
                neg = True
                i += 1
            comp = toks[i:]
            return self._complement(comp, subj, neg, ctx)
        if w in AUX:
            i = 1
            if i < len(words) and words[i] in NEGATORS:
                neg = True
                i += 1
            rest = toks[i:]
            if not rest:
                raise ParseFailure("dangling auxiliary")
            c = self._verb_phrase(rest, subj, ctx)
            if neg:
                c.lits = [(not p, q, a) for p, q, a in c.lits]
                c.negated = True
            return c
        if w in ("not", "never"):
            c = self._verb_phrase(toks[1:], subj, ctx)
            c.lits = [(not p, q, a) for p, q, a in c.lits]
            c.negated = True
            return c
        # Lexical verb.
        verb = lemma_verb(w)
        self._pending_verbs.add(w)
        rest = toks[1:]
        if not rest:
            return Clause(subj, [(True, verb, (subj,))], verb=w)
        rwords = [t.text for t in rest]
        if rwords[0] in PREPOSITIONS and len(rwords) > 1:
            verb = f"{verb}_{rwords[0]}"
            rest = rest[1:]
            rwords = rwords[1:]
        if rwords[0] in ("a", "an") or (rwords[0] not in ("the",) and not rest[0].cap
                                        and rwords[0] not in PRONOUNS | INDEFINITES
                                        and rwords[0] not in self.known_entities
                                        and len(rwords) > 1):
            # "has a tail", "plays tennis well" -> unary predicate
            content = [x for x in rwords if x not in ("a", "an", ",")]
            if any(x in COPULA | AUX | MODALS | RELATIVE for x in content):
                raise ParseFailure("clause inside verb object")
            pred = f"{verb}_" + "_".join(singular(x) for x in content)
            return Clause(subj, [(True, pred, (subj,))], verb=w)
        obj = self._object(rest, ctx)
        if obj is None:
            raise ParseFailure(f"cannot parse object {' '.join(rwords)!r}")
        term, _ = obj
        return Clause(subj, [(True, verb, (subj, term))], verb=w)

    def _complement(self, toks, subj, neg, ctx) -> Clause:
        words = [t.text for t in toks]
        if not words:
            raise ParseFailure("empty complement")
        if words[0] == "neither" and "nor" in words:
            k = words.index("nor")
            a = self._complement(toks[1:k], subj, False, ctx)
            b = self._complement(toks[k + 1 :], subj, False, ctx)
            lits = [(not p, q, ar) for p, q, ar in a.lits + b.lits]
            return Clause(subj, lits, copular=True)
        if words[0] == "either" and "or" in words:
            k = words.index("or")
            a = self._complement(toks[1:k], subj, False, ctx)
            b = self._complement(toks[k + 1 :], subj, False, ctx)
            if len(a.lits) == 1 and len(b.lits) == 1:
                return Clause(subj, a.lits + b.lits, kind="or", copular=True)
            raise ParseFailure("complex either/or")
        if "or" in words and words.index("or") > 0:
            k = words.index("or")
            a = self._complement(toks[:k], subj, neg, ctx)
            b = self._complement(toks[k + 1 :], subj, neg, ctx)
            if len(a.lits) == 1 and len(b.lits) == 1 and not neg:
                return Clause(subj, a.lits + b.lits, kind="or", copular=True)
            raise ParseFailure("complex disjunctive complement")
        if words[0] in ("a", "an"):
            content = [t for t in toks[1:] if t.text not in (",", "and")]
            if not content:
                raise ParseFailure("empty noun complement")
            rel = next((i for i, t in enumerate(content) if t.text in RELATIVE), None)
            extra = []
            if rel is not None:
                rc = self._conjunction(content[rel + 1 :], ctx, Clause(subj, []), subject=subj)
                for c in rc:
                    extra.extend(c.lits)
                content = content[:rel]
            pp = next((i for i, t in enumerate(content) if i > 0 and t.text in PREPOSITIONS), None)
            if pp is not None:
                obj = self._object(content[pp + 1 :], ctx) if pp + 1 < len(content) else None
                noun = singular(content[pp - 1].text)
                if obj is not None:
                    extra.append((True, f"{noun}_{content[pp].text}", (subj, obj[0])))
                content = content[:pp]
            lits = [(True, t.text, (subj,)) for t in content[:-1]]
            head = content[-1].text
            if head not in GENERIC_NOUNS or not lits:
                lits.append((True, singular(head), (subj,)))
            lits.extend(extra)
            if neg:
                if len(lits) != 1:
                    # ¬(a ∧ b): keep only the noun (the most specific claim).
                    lits = [lits[-1]] if not extra else lits[:1]
                lits = [(False, q, a) for _, q, a in lits]
            return Clause(subj, lits, copular=True, negated=neg, indefinite=True)
        if words[0] == "the" or (toks[0].cap and len(words) <= 3):
            obj = self._object(toks, ctx)
            if obj is not None:
                return Clause(subj, [(not neg, "=", (subj, obj[0]))], copular=True, negated=neg)
        # Adjective, optionally followed by a prepositional object: "afraid of the dog".
        content = [w for w in words if w != ","]
        if len(content) == 1:
            pred = singular(content[0]) if looks_plural(content[0]) else content[0]
            return Clause(subj, [(not neg, pred, (subj,))], copular=True, negated=neg)
        if len(content) >= 2 and content[1] in PREPOSITIONS:
            obj = self._object(toks[2:], ctx)
            if obj is not None:
                pred = f"{content[0]}_{content[1]}"
                return Clause(subj, [(not neg, pred, (subj, obj[0]))], copular=True, negated=neg)
        if all(w not in COPULA | AUX | DETERMINERS for w in content) and len(content) <= 4:
            # Multi-word adjective / plural noun complement ("are big dogs", "is well known").
            if looks_plural(content[-1]) and len(content) > 1:
                lits = [(not neg, w, (subj,)) for w in content[:-1]]
                lits.append((not neg, singular(content[-1]), (subj,)))
                return Clause(subj, lits, copular=True, negated=neg)
            return Clause(subj, [(not neg, "_".join(content), (subj,))], copular=True, negated=neg)
        raise ParseFailure(f"cannot parse complement {' '.join(words)!r}")


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------


def split_sentences(text: str) -> list[str]:
    text = text.replace("\n", " . ")
    parts = re.split(r"(?<=[.!?])\s+", text)
    return [p.strip() for p in parts if p.strip(" .")]


def _split_top(toks, word):
    groups, cur = [], []
    for t in toks:
        if t.text == word:
            if cur and cur[-1].text == ",":
                cur = cur[:-1]
            groups.append(cur)
            cur = []
        else:
            cur.append(t)
    groups.append(cur)
    return [g for g in groups if g]


def _split_coordination(toks):
    """Split on 'and' / ',' outside either..or / neither..nor complements."""
    words = [t.text for t in toks]
    if "either" in words or "neither" in words:
        # Only split on 'and' preceding the either/neither construction.
        k = words.index("either") if "either" in words else words.index("neither")
        head = _split_coordination(toks[:k]) if k > 0 else [[]]
        if head and head[-1] is not None:
            head[-1] = head[-1] + toks[k:]
        return head
    parts, cur = [], []
    for t in toks:
        if t.text in ("and", ",", "but"):
            if cur:
                parts.append(cur)
            cur = []
        else:
            cur.append(t)
    if cur:
        parts.append(cur)
    return parts


def _lit_formula(lit):
    positive, pred, args = lit
    atom = ("atom", pred, args)
    return atom if positive else ("not", atom)


def _vars_in_lits(lits):
    out = []
    for _, _, args in lits:
        for a in args:
            if type(a) is Var and a not in out:
                out.append(a)
    return out


def _implication(body_lits, head_clause):
    body = ("and", *(_lit_formula(lit) for lit in body_lits)) if body_lits else ("true",)
    head = ("or", *(_lit_formula(lit) for lit in head_clause.lits))
    f = ("imp", body, head)
    for v in _vars_in_lits(body_lits + head_clause.lits):
        f = ("all", v, f)
    return f


def _negated_universal(restrictor, clause):
    body = ("and", *(_lit_formula(lit) for lit in restrictor)) if restrictor else ("true",)
    inner = ("and", *(_lit_formula(lit) for lit in clause.lits))
    if clause.kind == "or":
        inner = ("or", *(_lit_formula(lit) for lit in clause.lits))
    f = ("imp", body, ("not", inner))
    for v in _vars_in_lits(restrictor + clause.lits):
        f = ("all", v, f)
    return f

## Implementation: `causal` — Counterfactual reasoning over Boolean structural causal models

In [6]:
"""Deterministic counterfactual reasoning over Boolean causal models.

Natural-language causal statements ("A causes B", "A and B together cause C",
"not A or B causes C", "A causes not B", "A has a direct effect on B") are
compiled into structural equations V := f(parents). Counterfactual queries
are answered with Pearl's three steps, implemented as exhaustive model
checking over the unobserved root (exogenous) variables:

1. Abduction  -- enumerate root assignments consistent with the factual
                 evidence (the antecedent's factual value and observations),
                 weighting each by its prior (``A ~ Bern(p)``; default 0.5).
2. Action     -- replace the equations of intervened variables by constants.
3. Prediction -- propagate in topological order and read off the outcome.

The answer is "yes" when the outcome holds with posterior probability > 0.5.
"""

import itertools
import re
from dataclasses import dataclass, field

CAUSE_VERBS = r"(?:causes?|caused|has a direct effect on|have direct effects on|" \
              r"has direct effects on|have a direct effect on|leads? to|results? in|produces?)"


@dataclass
class CausalModel:
    equations: dict = field(default_factory=dict)  # var -> expression
    priors: dict = field(default_factory=dict)  # root var -> P(var = 1)
    variables: list = field(default_factory=list)

    def add_var(self, v):
        if v not in self.variables:
            self.variables.append(v)

    def roots(self):
        return [v for v in self.variables if v not in self.equations]

    def order(self):
        done, out = set(), []

        def visit(v, stack=()):
            if v in done or v in stack:
                # A back edge (cyclic description): the parent is read with
                # its value from the earlier evaluation order (default False).
                return
            for p in _expr_vars(self.equations.get(v, ("const", False))):
                visit(p, stack + (v,))
            done.add(v)
            out.append(v)

        for v in self.variables:
            visit(v)
        return out

    def evaluate(self, roots: dict, interventions: dict) -> dict:
        world = {}
        for v in self.order():
            if v in interventions:
                world[v] = interventions[v]
            elif v in self.equations:
                world[v] = _eval(self.equations[v], world)
            else:
                world[v] = roots.get(v, False)
        return world


def _expr_vars(e):
    if e[0] == "var":
        return [e[1]]
    if e[0] == "const":
        return []
    if e[0] == "not":
        return _expr_vars(e[1])
    return [v for sub in e[1] for v in _expr_vars(sub)]


def _eval(e, world):
    op = e[0]
    if op == "var":
        return world.get(e[1], False)
    if op == "const":
        return e[1]
    if op == "not":
        return not _eval(e[1], world)
    if op == "and":
        return all(_eval(s, world) for s in e[1])
    return any(_eval(s, world) for s in e[1])


def expr_str(e):
    op = e[0]
    if op == "var":
        return e[1]
    if op == "const":
        return str(e[1])
    if op == "not":
        return "¬" + expr_str(e[1])
    sym = " ∧ " if op == "and" else " ∨ "
    return "(" + sym.join(expr_str(s) for s in e[1]) + ")"


# ---------------------------------------------------------------------------
# Parsing
# ---------------------------------------------------------------------------

_LITERAL = r"(?:not\s+)?[A-Za-z][\w\-]*"


def looks_causal(context: str, query: str) -> bool:
    text = f"{context} {query}".lower()
    return bool(re.search(r"\b(causes?|direct effects? on)\b", text)) and bool(
        re.search(r"\bwould\b.*\boccur\b|\bsuppose\b|\binstead of\b|\bassume\b", query.lower())
    )


def _parse_cause_expr(text: str):
    text = re.sub(r"\b(together|both|jointly|either)\b", " ", text).strip()
    text = re.sub(r"\s+", " ", text)
    ors = [p.strip() for p in re.split(r"\s+or\s+", text) if p.strip()]
    disj = []
    for part in ors:
        ands = [a.strip() for a in re.split(r"\s+and\s+", part) if a.strip()]
        conj = [_parse_lit(a) for a in ands]
        disj.append(conj[0] if len(conj) == 1 else ("and", conj))
    return disj[0] if len(disj) == 1 else ("or", disj)


def _parse_lit(text: str):
    text = text.strip()
    neg = False
    while text.lower().startswith("not "):
        neg = not neg
        text = text[4:].strip()
    name = text.strip(" .,")
    if not re.fullmatch(r"[A-Za-z][\w\-]*", name):
        raise ValueError(f"bad causal variable {text!r}")
    e = ("var", name)
    return ("not", e) if neg else e


def parse_causal_context(text: str) -> tuple[CausalModel, dict]:
    """Return the causal model and the observations stated in ``text``."""
    model = CausalModel()
    observations = {}
    text = text.replace("\n", " ")
    text = re.sub(r"^\s*(we know that|imagine[^:]*:|given that)\s*", "", text, flags=re.I)

    # Priors: "Blaf~ Bern(0.2)" / "Blaf ~ Bernoulli(0.2)"
    for m in re.finditer(r"([A-Za-z][\w\-]*)\s*~\s*Bern(?:oulli)?\s*\(\s*([\d.]+)\s*\)", text):
        model.priors[m.group(1)] = float(m.group(2))
    text = re.sub(r"[A-Za-z][\w\-]*\s*~\s*Bern(?:oulli)?\s*\(\s*[\d.]+\s*\)\.?", " ", text)

    # Observations: "We observed X", "We observed not X and Y"
    for m in re.finditer(r"\bwe (?:observed?|see|know)\s+(?:that\s+)?([^.]*)", text, flags=re.I):
        for part in re.split(r",|\band\b", m.group(1)):
            part = part.strip()
            if not part or re.search(CAUSE_VERBS, part):
                continue
            part = re.sub(r"\b(did|does|do)\s+not\b", "not", part)
            m_neg = re.match(r"^([A-Za-z][\w\-]*)\s+not\b(.*)$", part)
            if m_neg:
                part = f"not {m_neg.group(1)} {m_neg.group(2)}"
            part = re.sub(r"\b(occurred|occurs|occur|happened|happens|happen|is true)\b", "",
                          part)
            lit = _parse_lit(part)
            if lit[0] == "not":
                observations[lit[1][1]] = False
            else:
                observations[lit[1]] = True
    text = re.sub(r"\bwe (?:observed?|see)\b[^.]*\.?", " ", text, flags=re.I)

    # Causal statements, separated by commas / sentence ends / "and" before a new clause.
    text = re.sub(r"\.\s*", ", ", text)
    pieces = [p.strip() for p in re.split(r",", text) if p.strip()]
    statements = []
    for piece in pieces:
        piece = re.sub(r"^(and|also|then)\s+", "", piece.strip(), flags=re.I)
        if statements and not re.search(rf"\b{CAUSE_VERBS}\b", piece, flags=re.I):
            # Continuation of an effect list: "A causes B, C and D".
            statements[-1] += " and " + piece
            continue
        # Split "A causes B and C causes D" into two statements.
        verbs = list(re.finditer(rf"\b{CAUSE_VERBS}\b", piece, flags=re.I))
        if len(verbs) > 1:
            cut_points = []
            for v in verbs[1:]:
                before = piece[: v.start()]
                m = list(re.finditer(r"\band\b", before))
                cause_start = None
                # The cause of the second verb starts after the last 'and' that
                # follows the previous verb's effects.
                for a in reversed(m):
                    if a.start() > verbs[0].end():
                        cause_start = a
                        break
                if cause_start is not None:
                    cut_points.append(cause_start)
            last = 0
            for c in cut_points:
                statements.append(piece[last : c.start()])
                last = c.end()
            statements.append(piece[last:])
        else:
            statements.append(piece)

    for st in statements:
        st = st.strip()
        m = re.match(rf"^(.*?)\s+{CAUSE_VERBS}\s+(.*)$", st, flags=re.I)
        if not m:
            continue
        cause = _parse_cause_expr(m.group(1))
        effects = [e.strip() for e in re.split(r"\s+and\s+|\s*,\s*", m.group(2)) if e.strip()]
        for v in _expr_vars(cause):
            model.add_var(v)
        for eff in effects:
            lit = _parse_lit(eff)
            neg = lit[0] == "not"
            name = lit[1][1] if neg else lit[1]
            model.add_var(name)
            e = ("not", cause) if neg else cause
            if name in model.equations:
                model.equations[name] = ("or", [model.equations[name], e])
            else:
                model.equations[name] = e
    return model, observations


@dataclass
class CausalQuery:
    outcome: str
    outcome_value: bool = True
    interventions: dict = field(default_factory=dict)
    factual: dict = field(default_factory=dict)


def parse_causal_query(question: str) -> CausalQuery:
    q = question.strip().rstrip("?. ")
    # Nested: "Assume not A, and based on this assumption, further suppose not B. Would Y occur"
    m = re.search(r"would\s+(not\s+)?([A-Za-z][\w\-]*)\s+(not\s+)?(?:occur|happen|be true)", q,
                  flags=re.I)
    if not m:
        raise ValueError("unrecognised causal question")
    outcome = m.group(2)
    value = (m.group(1) is None) == (m.group(3) is None)
    q = q[: m.start()] + q[m.end():]
    query = CausalQuery(outcome, value)

    def lits(text):
        out = []
        for part in re.split(r"\s+and\s+|,", text):
            part = part.strip()
            if part:
                lit = _parse_lit(part)
                out.append((lit[1][1], False) if lit[0] == "not" else (lit[1], True))
        return out

    inst = re.search(r"\bif\s+(.*?)\s+instead of\s+(.*)$", q, flags=re.I)
    if inst:
        for v, val in lits(inst.group(1)):
            query.interventions[v] = val
        for v, val in lits(inst.group(2)):
            query.factual[v] = val
        return query
    for pattern in (r"\bassume\s+(.*?)(?:,|\.)", r"\bsuppose\s+(.*?)(?:\.|\s+would\b|$)",
                    r"\bif\s+(.*)$"):
        for mm in re.finditer(pattern, q, flags=re.I):
            text = re.sub(r"\b(and based on this assumption|further|that)\b", " ", mm.group(1))
            for v, val in lits(text):
                query.interventions[v] = val
    if not query.interventions:
        raise ValueError("no intervention found")
    # The intervened cause took the opposite value in the factual world.
    first = next(iter(query.interventions))
    query.factual.setdefault(first, not query.interventions[first])
    return query


# ---------------------------------------------------------------------------
# Inference
# ---------------------------------------------------------------------------


@dataclass
class CausalAnswer:
    answer: bool
    probability: float
    worlds: int
    trace: list


def counterfactual(model: CausalModel, query: CausalQuery, observations: dict) -> CausalAnswer:
    for v in list(query.interventions) + [query.outcome] + list(observations):
        model.add_var(v)
    roots = model.roots()
    # Observations constrain the exogenous (root) variables during abduction.
    # Observations of endogenous variables are already determined by the
    # structural equations given the roots, and the benchmark treats them as
    # background information rather than as evidence that could contradict the
    # antecedent; using them is only a fallback when roots are unconstrained.
    root_obs = {k: v for k, v in observations.items() if k not in model.equations}
    evidence_sets = [
        {**root_obs, **query.factual},
        dict(query.factual),
        {},
    ]
    trace = [f"{v} := {expr_str(e)}" for v, e in model.equations.items()]
    for evidence in evidence_sets:
        total = 0.0
        yes = 0.0
        worlds = 0
        for values in itertools.product((True, False), repeat=len(roots)):
            assignment = dict(zip(roots, values))
            factual = model.evaluate(assignment, {})
            if any(factual.get(k) != v for k, v in evidence.items()):
                continue
            weight = 1.0
            for r, val in assignment.items():
                p = model.priors.get(r, 0.5)
                weight *= p if val else 1 - p
            cf = model.evaluate(assignment, query.interventions)
            worlds += 1
            total += weight
            if cf[query.outcome] == query.outcome_value:
                yes += weight
            if worlds <= 2:
                trace.append(
                    "abduced roots "
                    + ", ".join(f"{k}={int(v)}" for k, v in assignment.items())
                    + " | do(" + ", ".join(f"{k}={int(v)}" for k, v in query.interventions.items())
                    + ") -> " + ", ".join(f"{k}={int(v)}" for k, v in cf.items())
                )
        if worlds:
            prob = yes / total if total else 0.0
            return CausalAnswer(prob > 0.5, prob, worlds, trace)
    return CausalAnswer(False, 0.0, 0, trace)


def answer_causal(context: str, question: str) -> CausalAnswer:
    model, observations = parse_causal_context(context)
    query = parse_causal_query(question)
    return counterfactual(model, query, observations)

## Implementation: `agent` — The reasoning agent: routing representation -> inference engine

In [7]:
"""End-to-end reasoning agent: (context, query) -> True / False.

Pipeline
    natural language / FOL text
        -> representation (Horn rules, FOL clauses, or a causal model)
        -> classical inference (forward/backward chaining, resolution,
           counterfactual model checking)
        -> label

Routing
    * FOL annotations (or FOL symbols in the context) -> resolution prover.
    * Causal language + counterfactual question     -> structural causal model.
    * English that compiles to Horn rules            -> forward chaining with
      negation as failure (closed-world, well-founded semantics).
    * English with disjunctive / existential content -> resolution prover
      (open-world).
"""

import re
import time
from collections import OrderedDict
from dataclasses import dataclass, field


FOL_SYMBOLS = re.compile(r"[∀∃→∧∨¬⊕↔]")


class _CompiledKB:
    """A parsed knowledge base plus lazily computed inference structures."""

    def __init__(self, parser: EnglishParser, items, failures):
        self.parser = parser
        self.failures = failures
        self.rules = [i for i in items if isinstance(i, Rule)]
        self.formulas = [i[1] for i in items if not isinstance(i, Rule)]
        self.controlled = not failures and EnglishParser.is_controlled(items)
        self.constants = {a for r in self.rules for _, _, args in (*r.body, r.head)
                          for a in args if type(a) is not Var}
        self.model = None
        self.chainer = None


@dataclass
class Prediction:
    label: bool
    engine: str
    trace: list = field(default_factory=list)
    seconds: float = 0.0
    detail: str = ""


class ReasoningAgent:
    def __init__(self, strategy: str = "forward", prover_time_limit: float = 0.3,
                 closed_world: bool = True, use_fol_annotations: bool = True,
                 cache_size: int = 4096):
        if strategy not in ("forward", "backward"):
            raise ValueError("strategy must be 'forward' or 'backward'")
        self.strategy = strategy
        self.closed_world = closed_world
        self.use_fol = use_fol_annotations
        self.prover = Prover(time_limit=prover_time_limit)
        self.cache_size = cache_size
        self._kb_cache: OrderedDict[str, _CompiledKB] = OrderedDict()

    # -- entry point --------------------------------------------------------------

    def predict(self, context: str, query: str, premises_fol: str | None = None,
                conclusion_fol: str | None = None, explain_proof: bool = False) -> Prediction:
        start = time.perf_counter()
        try:
            if self.use_fol and premises_fol and conclusion_fol:
                pred = self._fol(premises_fol, conclusion_fol)
            elif FOL_SYMBOLS.search(context or "") and FOL_SYMBOLS.search(query or "") or (
                FOL_SYMBOLS.search(context or "") and _looks_like_fol(query)
            ):
                pred = self._fol(context, query)
            elif looks_causal(context, query):
                pred = self._causal(context, query)
            else:
                pred = self._english(context, query, explain_proof)
        except Exception as exc:  # never crash on a single malformed example
            pred = Prediction(False, "fallback", [f"error: {type(exc).__name__}: {exc}"])
        pred.seconds = time.perf_counter() - start
        return pred

    # -- engines ------------------------------------------------------------------

    def _fol(self, premises_text: str, conclusion_text: str) -> Prediction:
        if not isinstance(premises_text, str):
            premises_text = "\n".join(premises_text)
        premises = []
        skipped = 0
        for line in re.split(r"\n|(?<=\))\s*;\s*", premises_text):
            if line.strip():
                try:
                    premises.append(parse_fol(line))
                except ValueError:
                    skipped += 1
        goal = parse_fol(conclusion_text, free_quantifier="ex")
        premises, goal, repairs = repair_symbols(premises, goal)
        result = prove_formula(premises, goal, self.prover)
        trace = [f"premises: {len(premises)} formulas ({skipped} unparsable)"]
        if repairs:
            trace.append("symbol repairs: " + ", ".join(f"{k}->{v}" for k, v in repairs.items()))
        trace.append(f"prover: {result.reason}, {result.steps} given clauses, "
                     f"{result.clauses} kept")
        trace += result.proof[-25:]
        return Prediction(result.proved, "resolution", trace, detail=result.reason)

    def _causal(self, context: str, query: str) -> Prediction:
        ans = answer_causal(context, query)
        trace = ans.trace + [f"P(outcome) = {ans.probability:.3f} over {ans.worlds} world(s)"]
        return Prediction(ans.answer, "causal", trace)

    def _compile(self, context: str) -> "_CompiledKB":
        """Parse a context once; repeated queries on the same KB reuse it."""
        kb = self._kb_cache.get(context)
        if kb is not None:
            self._kb_cache.move_to_end(context)
            return kb
        parser = EnglishParser()
        items, failures = parser.parse_text(context)
        parser.last_entity = None
        kb = _CompiledKB(parser, items, failures)
        self._kb_cache[context] = kb
        if len(self._kb_cache) > self.cache_size:
            self._kb_cache.popitem(last=False)
        return kb

    def _english(self, context: str, query: str, explain_proof: bool) -> Prediction:
        kb = self._compile(context)
        try:
            q_items = kb.parser.fork().parse_sentence(query)
        except ParseFailure as exc:
            return Prediction(False, "fallback", [f"query not understood: {exc}"])
        q_rules = [i for i in q_items if isinstance(i, Rule)]
        q_formulas = [i[1] for i in q_items if not isinstance(i, Rule)]
        notes = [f"parsed {len(kb.rules)} rules/facts, {len(kb.formulas)} formulas, "
                 f"{len(kb.failures)} unparsed sentences"]
        notes += [f"unparsed: {s!r} ({e})" for s, e in kb.failures[:5]]
        if not q_rules and not q_formulas:
            return Prediction(False, "fallback", notes + ["empty query"])

        if kb.controlled and EnglishParser.is_controlled(q_items) and self.closed_world:
            return self._horn(kb, q_rules, notes, explain_proof)
        notes.append("free-form text: open-world semantics")
        return self._open_world(kb.rules, kb.formulas, q_rules, q_formulas, notes)

    def _horn(self, kb, q_rules, notes, explain_proof) -> Prediction:
        rules = list(kb.rules)
        goals = []

        def sub(args):
            return tuple("hypothetical_individual" if type(a) is Var else a for a in args)

        hypothetical = False
        for q in q_rules:
            if q.body:
                # A rule-shaped query: hypothetically assert the body for a
                # fresh individual and test whether the head follows.
                rules += [Rule((), (p, n, sub(a))) for p, n, a in q.body]
                goals.append((q.head[0], q.head[1], sub(q.head[2])))
                hypothetical = True
            else:
                goals.append(q.head)
        if any(type(a) is Var for g in goals for a in g[2]):
            return Prediction(False, "fallback", notes + ["non-ground query"])

        domain = {a for g in goals for a in g[2]}
        # The cached model is valid when the query adds no rules and no new
        # individuals (negation as failure quantifies over the domain).
        reusable = not hypothetical and domain <= kb.constants
        if self.strategy == "backward":
            if reusable:
                kb.chainer = kb.chainer or BackwardChainer(rules, (), cwa=True)
                bc = kb.chainer
            else:
                bc = BackwardChainer(rules, domain, cwa=True)
            label = all(bc.prove(g) for g in goals)
            trace = notes + [f"backward chaining: {bc.calls} goal calls"]
            return Prediction(label, "backward-chaining", trace)

        if reusable:
            kb.model = kb.model or ForwardChainer(rules, (), cwa=True).model()
            model = kb.model
        else:
            model = ForwardChainer(rules, domain, cwa=True).model()
        label = all(model.holds(*g) for g in goals)
        trace = notes + [f"forward chaining: {len(model.true)} true atoms, "
                         f"{model.iterations} alternating-fixpoint round(s), "
                         f"{model.rule_firings} rule firings"]
        if explain_proof:
            for g in goals:
                trace += explain(model, g)
        else:
            trace += [f"query {literal_str(g)}: {'holds' if model.holds(*g) else 'fails'}"
                      for g in goals]
        return Prediction(label, "forward-chaining", trace)

    def _open_world(self, rules, formulas, q_rules, q_formulas, notes) -> Prediction:
        premises = [_rule_formula(r) for r in rules] + [close_universally(f) for f in formulas]
        parts = [_rule_formula(r) for r in q_rules] + [close_universally(f) for f in q_formulas]
        goal = parts[0] if len(parts) == 1 else ("and", *parts)
        result = prove_formula(premises, goal, self.prover)
        trace = notes + [f"resolution: {result.reason}"] + result.proof[-15:]
        return Prediction(result.proved, "resolution", trace, detail=result.reason)


def _looks_like_fol(text: str) -> bool:
    return bool(re.match(r"^\s*¬?\s*[A-Za-z]\w*\([\w\s,]+\)\s*$", text or ""))


def _rule_formula(rule: Rule):
    def lit(literal):
        positive, pred, args = literal
        atom = ("atom", pred, args)
        return atom if positive else ("not", atom)

    head = lit(rule.head)
    f = head if not rule.body else ("imp", ("and", *(lit(b) for b in rule.body)), head)
    return close_universally(f)


def explain_text(agent: ReasoningAgent, context: str, query: str, max_context=300) -> str:
    pred = agent.predict(context, query, explain_proof=True)
    shown = " ".join(split_sentences(context))
    if max_context and len(shown) > max_context:
        shown = shown[:max_context] + " ..."
    lines = [f"Context: {shown}", f"Query: {query}",
             f"Engine: {pred.engine}", f"Prediction: {pred.label}"]
    lines += ["  " + t for t in pred.trace]
    return "\n".join(lines)

## Implementation: `data` — Dataset loading (RuleTaker, FOLIO, CounterBench, unseen test files)

In [8]:
"""Dataset loading: RuleTaker, FOLIO, CounterBench and arbitrary test files.

Every example is normalised to a dict with keys
    id, source, context, query, label (bool or None),
    premises_fol, conclusion_fol (optional), meta
"""

import csv
import json
import os
from pathlib import Path


def _default_data_dir() -> Path:
    if os.environ.get("REASONER_DATA_DIR"):
        return Path(os.environ["REASONER_DATA_DIR"])
    try:
        return Path(__file__).resolve().parents[2] / "data" / "raw"
    except NameError:  # running inside the notebook
        for candidate in (Path("data/raw"), Path("../data/raw")):
            if candidate.exists():
                return candidate
        return Path("data/raw")


RAW = _default_data_dir()


def ensure_public_data(raw_dir: Path = RAW) -> None:
    """Download the public datasets from Hugging Face if they are missing."""
    raw_dir.mkdir(parents=True, exist_ok=True)
    needed = {
        "ruletaker": ("tasksource/ruletaker", ("train", "dev", "test")),
        "folio": ("tasksource/folio", ("train", "validation")),
    }
    for short, (repo, splits) in needed.items():
        if all((raw_dir / f"{short}_{s}.jsonl").exists() for s in splits):
            continue
        from datasets import load_dataset

        ds = load_dataset(repo)
        for s in splits:
            ds[s].to_json(str(raw_dir / f"{short}_{s}.jsonl"))
    for name in ("data_balanced_alpha_V1.json", "data_balanced_backdoor_V2.json"):
        target = raw_dir / f"counterbench_{name}"
        if not target.exists():
            import shutil

            from huggingface_hub import hf_hub_download

            path = hf_hub_download("CounterBench/CounterBench", name, repo_type="dataset")
            shutil.copy(path, target)

CONTEXT_KEYS = ("context", "theory", "premises", "given_info", "kb", "knowledge_base",
                "facts", "passage", "story")
QUERY_KEYS = ("statement", "question", "query", "conclusion", "hypothesis", "claim")
LABEL_KEYS = ("flag", "label", "answer", "gold", "target", "truth")
ID_KEYS = ("id", "example_id", "question_id", "uid", "idx")


def to_bool(value):
    if value is None:
        return None
    if isinstance(value, bool):
        return value
    v = str(value).strip().lower()
    if v in ("true", "yes", "entailment", "entailed", "1", "t", "y"):
        return True
    if v in ("false", "no", "not entailment", "not_entailment", "0", "uncertain", "unknown",
             "f", "n", "contradiction", "neutral"):
        return False
    return None


def _read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def load_ruletaker(split="dev", limit=None, seed=0):
    rows = _read_jsonl(RAW / f"ruletaker_{split}.jsonl")
    if limit:
        import random

        rows = random.Random(seed).sample(rows, min(limit, len(rows)))
    return [
        {"id": f"ruletaker-{split}-{i}", "source": "ruletaker", "context": r["context"],
         "query": r["question"], "label": r["label"] == "entailment",
         "meta": {"config": r["config"], "raw_label": r["label"]}}
        for i, r in enumerate(rows)
    ]


def load_folio(split="validation"):
    rows = _read_jsonl(RAW / f"folio_{split}.jsonl")
    return [
        {"id": f"folio-{split}-{r['example_id']}", "source": "folio", "context": r["premises"],
         "query": r["conclusion"], "label": r["label"] == "True",
         "premises_fol": r["premises-FOL"], "conclusion_fol": r["conclusion-FOL"],
         "meta": {"raw_label": r["label"], "story_id": r["story_id"]}}
        for r in rows
    ]


def load_counterbench():
    out = []
    for name in ("data_balanced_alpha_V1", "data_balanced_backdoor_V2"):
        path = RAW / f"counterbench_{name}.json"
        for r in json.loads(path.read_text(encoding="utf-8")):
            out.append({
                "id": f"counterbench-{name}-{r['question_id']}", "source": "counterbench",
                "context": r["given_info"], "query": r["question"],
                "label": r["answer"].strip().lower() == "yes",
                "meta": {"type": r["type"], "raw_label": r["answer"]},
            })
    return out


def _pick(record, keys):
    lowered = {k.lower().strip(): k for k in record}
    for key in keys:
        if key in lowered:
            return record[lowered[key]]
    return None


def _as_text(value, sep=" "):
    if value is None:
        return ""
    if isinstance(value, str) and value.startswith("[") and value.endswith("]"):
        import ast

        try:  # a list serialised into a CSV cell
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            pass
    if not isinstance(value, str) and hasattr(value, "__iter__"):
        return sep.join(str(v).strip() for v in value)
    return str(value)


def normalize_record(record: dict, index: int) -> dict:
    """Normalise a record from an unseen test file with unknown column names."""
    context = _as_text(_pick(record, CONTEXT_KEYS))
    query = _as_text(_pick(record, QUERY_KEYS))
    rid = _pick(record, ID_KEYS)
    premises_fol = _pick(record, ("premises-fol", "premises_fol", "fol_premises"))
    conclusion_fol = _pick(record, ("conclusion-fol", "conclusion_fol", "fol_conclusion"))
    return {
        "id": str(rid) if rid is not None else str(index),
        "source": str(_pick(record, ("source", "dataset", "config")) or "unknown"),
        "context": context,
        "query": query,
        "label": to_bool(_pick(record, LABEL_KEYS)),
        "premises_fol": _as_text(premises_fol, sep="\n") or None,
        "conclusion_fol": _as_text(conclusion_fol) or None,
        "meta": {},
    }


def load_any(path) -> list[dict]:
    """Load a .csv / .json / .jsonl / .parquet test file."""
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        with open(path, encoding="utf-8", newline="") as f:
            rows = list(csv.DictReader(f))
    elif suffix == ".jsonl":
        rows = _read_jsonl(path)
    elif suffix == ".json":
        data = json.loads(path.read_text(encoding="utf-8"))
        rows = data if isinstance(data, list) else data.get("data", [data])
    elif suffix == ".parquet":
        import pandas as pd

        rows = pd.read_parquet(path).to_dict("records")
    else:
        raise ValueError(f"unsupported file type: {suffix}")
    return [normalize_record(r, i) for i, r in enumerate(rows)]

## Implementation: `evaluate` — Evaluation helpers and submission writer

In [9]:
"""Evaluation helpers: accuracy, timing, engine usage, and submission files."""

import csv
import time
from collections import Counter, defaultdict



def run(agent: ReasoningAgent, examples, use_fol=True, progress_every=0):
    """Predict every example. Returns a list of (example, Prediction)."""
    out = []
    for i, ex in enumerate(examples, 1):
        pred = agent.predict(
            ex["context"], ex["query"],
            ex.get("premises_fol") if use_fol else None,
            ex.get("conclusion_fol") if use_fol else None,
        )
        out.append((ex, pred))
        if progress_every and i % progress_every == 0:
            print(f"  {i}/{len(examples)}")
    return out


def summarize(results, title=""):
    by_source = defaultdict(Counter)
    engines = Counter()
    total_time = 0.0
    for ex, pred in results:
        total_time += pred.seconds
        engines[pred.engine] += 1
        if ex["label"] is None:
            continue
        key = ex["source"]
        by_source[key]["n"] += 1
        by_source[key]["correct"] += int(pred.label == ex["label"])
    n = sum(c["n"] for c in by_source.values())
    correct = sum(c["correct"] for c in by_source.values())
    lines = [f"== {title} ==" if title else "== results =="]
    for source, c in sorted(by_source.items()):
        lines.append(f"{source:>14}: {c['correct']}/{c['n']} = {100 * c['correct'] / c['n']:.2f}%")
    if n:
        lines.append(f"{'overall':>14}: {correct}/{n} = {100 * correct / n:.2f}%")
    lines.append(f"total inference time: {total_time:.3f}s "
                 f"({1000 * total_time / max(1, len(results)):.3f} ms/example)")
    lines.append("engines: " + ", ".join(f"{k}={v}" for k, v in engines.most_common()))
    return "\n".join(lines), (100 * correct / n if n else None), total_time


def breakdown(results, key):
    table = defaultdict(Counter)
    for ex, pred in results:
        if ex["label"] is None:
            continue
        k = key(ex)
        table[k]["n"] += 1
        table[k]["correct"] += int(pred.label == ex["label"])
    return {k: (v["correct"], v["n"]) for k, v in sorted(table.items())}


def write_submission(results, path="submission.csv", label_column="label"):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", label_column])
        for ex, pred in results:
            writer.writerow([ex["id"], "True" if pred.label else "False"])


def timed(fn, *args, **kwargs):
    start = time.perf_counter()
    value = fn(*args, **kwargs)
    return value, time.perf_counter() - start

## Public data
The raw files are read from `data/raw/` (downloaded from Hugging Face on first use).

In [10]:
ensure_public_data(RAW)
ruletaker = load_ruletaker(RULETAKER_SPLIT, RULETAKER_LIMIT)
folio_val = load_folio("validation")
folio_train = load_folio("train")
counterbench = load_counterbench()
print(f"RuleTaker {RULETAKER_SPLIT}: {len(ruletaker)} | FOLIO train/validation: "
      f"{len(folio_train)}/{len(folio_val)} | CounterBench: {len(counterbench)}")

RuleTaker test: 151911 | FOLIO train/validation: 1001/203 | CounterBench: 1200


## Experiment 1 — accuracy and inference time on the public datasets

In [11]:
agent = ReasoningAgent(strategy="forward", prover_time_limit=PROVER_TIME_LIMIT)
all_results = []
for name, examples in [("RuleTaker", ruletaker), ("FOLIO (validation)", folio_val),
                       ("FOLIO (train)", folio_train), ("CounterBench", counterbench)]:
    results = run(agent, examples)
    all_results += results
    print(summarize(results, name)[0], end="\n\n")
text, accuracy, total_time = summarize(all_results, "ALL PUBLIC DATA")
print(text)

== RuleTaker ==
     ruletaker: 151911/151911 = 100.00%
       overall: 151911/151911 = 100.00%
total inference time: 14.326s (0.094 ms/example)
engines: forward-chaining=151911



== FOLIO (validation) ==
         folio: 178/203 = 87.68%
       overall: 178/203 = 87.68%
total inference time: 11.939s (58.813 ms/example)
engines: resolution=203



== FOLIO (train) ==
         folio: 819/1001 = 81.82%
       overall: 819/1001 = 81.82%
total inference time: 44.255s (44.211 ms/example)
engines: resolution=1001

== CounterBench ==
  counterbench: 1140/1200 = 95.00%
       overall: 1140/1200 = 95.00%
total inference time: 0.153s (0.127 ms/example)
engines: causal=1200



== ALL PUBLIC DATA ==
  counterbench: 1140/1200 = 95.00%
         folio: 997/1204 = 82.81%
     ruletaker: 151911/151911 = 100.00%
       overall: 154048/154315 = 99.83%
total inference time: 70.672s (0.458 ms/example)
engines: forward-chaining=151911, resolution=1204, causal=1200


In [12]:
print("RuleTaker by depth / config:")
for k, (c, n) in breakdown([r for r in all_results if r[0]["source"] == "ruletaker"],
                           lambda e: e["meta"]["config"]).items():
    print(f"  {k:>20}: {c}/{n} = {100 * c / n:.2f}%")
print("CounterBench by question type:")
for k, (c, n) in breakdown([r for r in all_results if r[0]["source"] == "counterbench"],
                           lambda e: e["meta"]["type"]).items():
    print(f"  {k:>20}: {c}/{n} = {100 * c / n:.2f}%")
print("FOLIO by gold 3-way label (True = entailed, False/Uncertain = not entailed):")
for k, (c, n) in breakdown([r for r in all_results if r[0]["source"] == "folio"],
                           lambda e: e["meta"]["raw_label"]).items():
    print(f"  {k:>20}: {c}/{n} = {100 * c / n:.2f}%")

RuleTaker by depth / config:
               NatLang: 7793/7793 = 100.00%
               depth-0: 17347/17347 = 100.00%
               depth-1: 17788/17788 = 100.00%
               depth-2: 17741/17741 = 100.00%
               depth-3: 17982/17982 = 100.00%
            depth-3ext: 23222/23222 = 100.00%
    depth-3ext-NatLang: 31017/31017 = 100.00%
               depth-5: 19021/19021 = 100.00%
CounterBench by question type:
              backdoor: 180/200 = 90.00%
                 basic: 250/250 = 100.00%
           conditional: 230/250 = 92.00%
                 joint: 240/250 = 96.00%
                nested: 240/250 = 96.00%
FOLIO by gold 3-way label (True = entailed, False/Uncertain = not entailed):
                 False: 325/351 = 92.59%
                  True: 292/460 = 63.48%
             Uncertain: 380/393 = 96.69%


## Experiment 2 — forward vs. backward chaining (RuleTaker sample)

In [13]:
sample = ruletaker[:5000]
for strategy in ("forward", "backward"):
    res = run(ReasoningAgent(strategy=strategy), sample)
    print(summarize(res, f"{strategy} chaining")[0], end="\n\n")

== forward chaining ==
     ruletaker: 5000/5000 = 100.00%
       overall: 5000/5000 = 100.00%
total inference time: 0.861s (0.172 ms/example)
engines: forward-chaining=5000



== backward chaining ==
     ruletaker: 5000/5000 = 100.00%
       overall: 5000/5000 = 100.00%
total inference time: 0.379s (0.076 ms/example)
engines: backward-chaining=5000



## Experiment 3 — FOLIO without FOL annotations (English parser only)
Shows what the system does when only natural language is available.

In [14]:
res = run(agent, folio_val, use_fol=False)
print(summarize(res, "FOLIO validation, natural language only")[0])

== FOLIO validation, natural language only ==
         folio: 130/203 = 64.04%
       overall: 130/203 = 64.04%
total inference time: 8.819s (43.441 ms/example)
engines: resolution=139, fallback=59, forward-chaining=5


## Experiment 4 — closed-world vs. open-world semantics (RuleTaker sample)
RuleTaker labels follow negation-as-failure; reading `not` as classical negation (open world) leaves many negative queries unprovable.

In [15]:
for cwa in (True, False):
    res = run(ReasoningAgent(closed_world=cwa), sample)
    print(summarize(res, "closed world" if cwa else "open world")[0], end="\n\n")

== closed world ==
     ruletaker: 5000/5000 = 100.00%
       overall: 5000/5000 = 100.00%
total inference time: 0.460s (0.092 ms/example)
engines: forward-chaining=5000



== open world ==
     ruletaker: 3742/5000 = 74.84%
       overall: 3742/5000 = 74.84%
total inference time: 1.130s (0.226 ms/example)
engines: resolution=5000



## Experiment 5 — resolution time budget (FOLIO validation)

In [16]:
for limit in (0.05, 0.3, 1.0):
    res = run(ReasoningAgent(prover_time_limit=limit), folio_val)
    print(summarize(res, f"time limit {limit}s per proof")[0], end="\n\n")

== time limit 0.05s per proof ==
         folio: 177/203 = 87.19%
       overall: 177/203 = 87.19%
total inference time: 3.675s (18.102 ms/example)
engines: resolution=203



== time limit 0.3s per proof ==
         folio: 178/203 = 87.68%
       overall: 178/203 = 87.68%
total inference time: 11.845s (58.348 ms/example)
engines: resolution=203



== time limit 1.0s per proof ==
         folio: 178/203 = 87.68%
       overall: 178/203 = 87.68%
total inference time: 32.023s (157.747 ms/example)
engines: resolution=203



## Reasoning traces

In [17]:
example_kb = ("Bob is young. Bob is kind. If someone is young, then they are smart. "
              "If someone is smart and kind, then they are successful.")
print(explain_text(agent, example_kb, "Bob is successful."))
print()
# A RuleTaker query whose proof chains at least three rules.
ex = next(e for e in ruletaker if e["label"] and e["meta"]["config"].startswith("depth-3")
          and explain_text(agent, e["context"], e["query"]).count("[rule:") >= 3)
print(explain_text(agent, ex["context"], ex["query"]))
print()
# A query that is not entailed (closed-world assumption).
ex = next(e for e in ruletaker if not e["label"] and e["meta"]["config"] == "depth-2")
print(explain_text(agent, ex["context"], ex["query"]))

Context: Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful.
Query: Bob is successful.
Engine: forward-chaining
Prediction: True
  parsed 4 rules/facts, 0 formulas, 0 unparsed sentences
  forward chaining: 4 true atoms, 1 alternating-fixpoint round(s), 3 rule firings
  successful(bob)   [rule: smart(X) ∧ kind(X) → successful(X)]
    smart(bob)   [rule: young(X) → smart(X)]
      young(bob)   [fact]
    kind(bob)   [fact]

Context: Bob is young. Dave is quiet. Dave is smart. Dave is not white. Erin is white. Fiona is not red. Fiona is smart. If something is red then it is not nice. If Dave is nice then Dave is smart. If Fiona is kind and Fiona is not nice then Fiona is young. If something is young and quiet then it is red. If  ...
Query: Bob is not nice.
Engine: forward-chaining
Prediction: True
  parsed 15 rules/facts, 0 formulas, 0 unparsed sentences
  forward chaining: 9 true atoms, 3 alternating-fixpoint round(

In [18]:
def folio_pred(e):
    return agent.predict(e["context"], e["query"], e["premises_fol"], e["conclusion_fol"])

# A FOLIO conclusion proved by a short resolution refutation.
ex = next(e for e in folio_val if e["meta"]["raw_label"] == "True"
          and folio_pred(e).label and 8 <= len(folio_pred(e).trace) <= 16)
pred = folio_pred(ex)
print(ex["premises_fol"]); print("Conclusion:", ex["conclusion_fol"])
print("Prediction:", pred.label)
print("\n".join(pred.trace))
print()
ex = counterbench[0]
pred = agent.predict(ex["context"], ex["query"])
print(ex["context"]); print(ex["query"], "->", pred.label, "| gold:", ex["label"])
print("\n".join(pred.trace))

Game(theLegendofZelda) ∧ ∃x (Japanese(x) ∧ VideoGameCompany(x) ∧ Created(x, theLegendofZelda))
∀x ∀y ((Game(x) ∧ InTop10(x) ∧ Created(y,x)) → Japanese(y))
∀x ((Game(x) ∧ ∃y(GreaterThan(y, oneMillion) ∧ CopiesSold(x, y))) → Top10(x)))
∃y(GreaterThan(y, oneMillion) ∧ CopiesSold(theLegendofZelda,y))
Conclusion: Top10(thelegendofzelda)
Prediction: True
premises: 4 formulas (0 unparsable)
symbol repairs: thelegendofzelda->theLegendofZelda
prover: refutation found, 4 given clauses, 12 kept
[4] ¬Top10(theLegendofZelda)    <- goal
[1] ¬CopiesSold(X, Y) ∨ ¬Game(X) ∨ ¬GreaterThan(Y, oneMillion) ∨ Top10(X)    <- axiom
[5] ¬CopiesSold(theLegendofZelda, X) ∨ ¬Game(theLegendofZelda) ∨ ¬GreaterThan(X, oneMillion)    <- res(4, 1)
[3] CopiesSold(theLegendofZelda, sk1)    <- axiom
[6] ¬Game(theLegendofZelda) ∨ ¬GreaterThan(sk1, oneMillion)    <- res(5, 3)
[2] GreaterThan(sk1, oneMillion)    <- axiom
[10] ¬Game(theLegendofZelda)    <- res(6, 2)
[0] Game(theLegendofZelda)    <- axiom
[11] □    <- res(10, 

## Submission

In [19]:
if TEST_PATH.exists():
    test_examples = load_any(TEST_PATH)
    print(f"Predicting {len(test_examples)} examples from {TEST_PATH}")
    test_results = run(agent, test_examples)
else:
    print(f"{TEST_PATH} not found: writing predictions for the public evaluation sets")
    test_results = [r for r in all_results]
write_submission(test_results, SUBMISSION_PATH)
text, accuracy, total_time = summarize(test_results, "submission set")
print(text)
print(f"\nFINAL ACCURACY: {accuracy:.2f}%" if accuracy is not None
      else "\nFINAL ACCURACY: n/a (no gold labels)")
print(f"TOTAL INFERENCE TIME: {total_time:.2f} s for {len(test_results)} queries")
print(f"wrote {SUBMISSION_PATH.resolve()}")

test.csv not found: writing predictions for the public evaluation sets


== submission set ==
  counterbench: 1140/1200 = 95.00%
         folio: 997/1204 = 82.81%
     ruletaker: 151911/151911 = 100.00%
       overall: 154048/154315 = 99.83%
total inference time: 70.672s (0.458 ms/example)
engines: forward-chaining=151911, resolution=1204, causal=1200

FINAL ACCURACY: 99.83%
TOTAL INFERENCE TIME: 70.67 s for 154315 queries
wrote /home/mayu/Documents/Assignments/classical-ai-reasoning-agent-updated/submission/submission.csv
